In [ ]:
# ============================================================
# GOVERNANCE + STRATEGY SECTION GENERATORS
# Role-based Azure OpenAI REST endpoints
# Writer: GPT-5.1 | Judge: GPT-5.2 (LLM-only evaluation) | Reviser: GPT-4.1
# ============================================================

import os
import json
import re
import urllib.request
import urllib.error
import time
import random
from typing import TypedDict, Literal
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langgraph.graph import StateGraph, START, END

# ── ENV LOADING ──────────────────────────────────────────────
env_path = find_dotenv()
if env_path:
    load_dotenv(env_path, override=True)
    print(f"Loaded .env from: {env_path}")
else:
    load_dotenv(override=True)
    print("No .env found by find_dotenv(); using existing environment variables.")

# Shared-key fallback.
# When all three deployments belong to the same Azure resource, one Azure key
# can authenticate all roles. The code also accepts any existing role-specific
# key as the shared fallback, which prevents unnecessary configuration failures.
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")

_shared_key_fallback = (
    AZURE_OPENAI_API_KEY
    or os.getenv("AZURE_OPENAI_WRITER_API_KEY")
    or os.getenv("AZURE_OPENAI_JUDGE_API_KEY")
    or os.getenv("AZURE_OPENAI_REVISER_API_KEY")
)

AZURE_OPENAI_WRITER_API_KEY = (
    os.getenv("AZURE_OPENAI_WRITER_API_KEY")
    or _shared_key_fallback
)
AZURE_OPENAI_JUDGE_API_KEY = (
    os.getenv("AZURE_OPENAI_JUDGE_API_KEY")
    or _shared_key_fallback
)
AZURE_OPENAI_REVISER_API_KEY = (
    os.getenv("AZURE_OPENAI_REVISER_API_KEY")
    or _shared_key_fallback
)

# Full Azure chat-completions deployment URLs.
# AZURE_OPENAI_CHAT_URL is retained as a backward-compatible writer fallback.
AZURE_OPENAI_WRITER_URL = (
    os.getenv("AZURE_OPENAI_WRITER_URL")
    or os.getenv("AZURE_OPENAI_CHAT_URL")
)
AZURE_OPENAI_JUDGE_URL = os.getenv("AZURE_OPENAI_JUDGE_URL")
AZURE_OPENAI_REVISER_URL = os.getenv("AZURE_OPENAI_REVISER_URL")


def _clean_url(value: str | None) -> str | None:
    if not value:
        return None
    return value.strip().strip('"').strip("'")


AZURE_OPENAI_WRITER_URL = _clean_url(AZURE_OPENAI_WRITER_URL)
AZURE_OPENAI_JUDGE_URL = _clean_url(AZURE_OPENAI_JUDGE_URL)
AZURE_OPENAI_REVISER_URL = _clean_url(AZURE_OPENAI_REVISER_URL)


def validate_role_config() -> None:
    required = {
        "AZURE_OPENAI_WRITER_API_KEY": AZURE_OPENAI_WRITER_API_KEY,
        "AZURE_OPENAI_JUDGE_API_KEY": AZURE_OPENAI_JUDGE_API_KEY,
        "AZURE_OPENAI_REVISER_API_KEY": AZURE_OPENAI_REVISER_API_KEY,
        "AZURE_OPENAI_WRITER_URL": AZURE_OPENAI_WRITER_URL,
        "AZURE_OPENAI_JUDGE_URL": AZURE_OPENAI_JUDGE_URL,
        "AZURE_OPENAI_REVISER_URL": AZURE_OPENAI_REVISER_URL,
    }

    missing = [name for name, value in required.items() if not value]
    if missing:
        loaded_flags = {
            "shared_key_loaded": bool(AZURE_OPENAI_API_KEY),
            "writer_key_loaded": bool(AZURE_OPENAI_WRITER_API_KEY),
            "judge_key_loaded": bool(AZURE_OPENAI_JUDGE_API_KEY),
            "reviser_key_loaded": bool(AZURE_OPENAI_REVISER_API_KEY),
            "writer_url_loaded": bool(AZURE_OPENAI_WRITER_URL),
            "judge_url_loaded": bool(AZURE_OPENAI_JUDGE_URL),
            "reviser_url_loaded": bool(AZURE_OPENAI_REVISER_URL),
        }
        raise ValueError(
            "Missing role-based Azure configuration values: "
            + ", ".join(missing)
            + "\n\nLoaded configuration flags (keys are never printed):\n"
            + json.dumps(loaded_flags, indent=2)
            + "\n\nRequired .env configuration when all deployments use the same Azure resource:\n"
              "AZURE_OPENAI_API_KEY=<shared Azure resource key>\n"
              "AZURE_OPENAI_WRITER_URL=<full GPT-5.1 deployment URL>\n"
              "AZURE_OPENAI_JUDGE_URL=<full GPT-5.2 deployment URL>\n"
              "AZURE_OPENAI_REVISER_URL=<full GPT-4.1 deployment URL>\n\n"
              "Use role-specific API keys only when a deployment belongs to a different Azure resource."
        )

    for name, url in {
        "AZURE_OPENAI_WRITER_URL": AZURE_OPENAI_WRITER_URL,
        "AZURE_OPENAI_JUDGE_URL": AZURE_OPENAI_JUDGE_URL,
        "AZURE_OPENAI_REVISER_URL": AZURE_OPENAI_REVISER_URL,
    }.items():
        if not url.startswith("https://"):
            raise ValueError(f"{name} must be a full HTTPS Azure deployment URL: {url!r}")


validate_role_config()

print("Role-based Azure OpenAI configuration loaded")
print("Writer endpoint (GPT-5.1):", AZURE_OPENAI_WRITER_URL[:90] + "...")
print("Judge endpoint  (GPT-5.2):", AZURE_OPENAI_JUDGE_URL[:90] + "...")
print("Reviser endpoint (GPT-4.1):", AZURE_OPENAI_REVISER_URL[:90] + "...")


def _azure_chat_completion(
    *,
    url: str,
    api_key: str,
    messages: list[dict],
    max_output_tokens: int,
    json_mode: bool = False,
    temperature: float | None = None,
    use_max_completion_tokens: bool = False,
    timeout: int = 240,
    request_label: str = "LLM",
    max_attempts: int = 4,
) -> dict:
    """
    Robust REST call for Azure/OpenAI-compatible enterprise gateways.

    Behaviour:
    - Retries transient 500/502/503/504 and connection errors.
    - For GPT-5.x gateways, first tries `max_completion_tokens`.
    - If the gateway returns 400/500, retries using `max_tokens`, because some
      enterprise proxies do not yet forward `max_completion_tokens` correctly.
    - Does not expose API keys in errors.
    """

    preferred_field = (
        "max_completion_tokens" if use_max_completion_tokens else "max_tokens"
    )
    token_fields = [preferred_field]
    if preferred_field == "max_completion_tokens":
        token_fields.append("max_tokens")

    last_error = None

    for token_field in token_fields:
        payload = {
            "messages": messages,
            token_field: max_output_tokens,
        }

        if temperature is not None:
            payload["temperature"] = temperature

        if json_mode:
            payload["response_format"] = {"type": "json_object"}

        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(
                url,
                data=json.dumps(payload).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "api-key": api_key,
                },
                method="POST",
            )

            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    return json.loads(resp.read().decode("utf-8"))

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"{request_label} HTTP error {exc.code}.\n"
                    f"Endpoint: {url}\n"
                    f"Token field used: {token_field}\n"
                    f"Response: {body[:3000]}"
                )

                transient = exc.code in {500, 502, 503, 504}
                compatibility_candidate = (
                    token_field == "max_completion_tokens"
                    and exc.code in {400, 422, 500}
                )

                if transient and attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: server error {exc.code}; "
                        f"retrying attempt {attempt + 1}/{max_attempts} "
                        f"in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                if compatibility_candidate:
                    print(
                        f"{request_label}: gateway may not support "
                        "`max_completion_tokens`; retrying with `max_tokens`."
                    )
                    break

                raise last_error from exc

            except urllib.error.URLError as exc:
                last_error = RuntimeError(
                    f"{request_label} connection error.\n"
                    f"Endpoint: {url!r}\n"
                    f"Error: {exc}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection issue; retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                raise last_error from exc

    raise last_error or RuntimeError(
        f"{request_label} request failed for an unknown reason."
    )

def _extract_message_content(data: dict) -> str:
    try:
        content = data["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise ValueError(
            "Unexpected Azure response structure:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        ) from exc

    if not content:
        raise ValueError(
            "Azure returned an empty message content. Response:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        )
    return content


def call_writer_llm(system_prompt: str, user_prompt: str) -> str:
    """GPT-5.1 writer."""
    data = _azure_chat_completion(
        url=AZURE_OPENAI_WRITER_URL,
        api_key=AZURE_OPENAI_WRITER_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=2400,
        use_max_completion_tokens=True,
        temperature=None,
        request_label="GPT-5.1 writer",
    )
    return _extract_message_content(data)


def _extract_json_object(text: str) -> str:
    """
    Extract the outermost JSON object from model output.
    Handles accidental markdown fences or leading/trailing commentary.
    """
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)

    first = text.find("{")
    last = text.rfind("}")
    if first >= 0 and last > first:
        return text[first:last + 1]

    return text


def call_judge_llm_json(system_prompt: str, user_prompt: str) -> dict:
    """
    GPT-5.2 judge with JSON-safe retry logic.

    First call:
    - Requests valid JSON mode.
    - Uses a larger output allowance to avoid truncation.

    On invalid/truncated JSON:
    - Sends the returned content back to GPT-5.2 for JSON repair.
    - Requests a concise, complete JSON object only.
    """
    data = _azure_chat_completion(
        url=AZURE_OPENAI_JUDGE_URL,
        api_key=AZURE_OPENAI_JUDGE_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=2800,
        use_max_completion_tokens=True,
        temperature=None,
        json_mode=True,
        request_label="GPT-5.2 judge",
    )

    content = _extract_message_content(data)
    candidate = _extract_json_object(content)

    try:
        return json.loads(candidate)

    except json.JSONDecodeError:
        print("GPT-5.2 judge returned incomplete/invalid JSON. Attempting JSON repair...")

        repair_system = (
            "You repair malformed or truncated JSON. "
            "Return one complete valid JSON object only. "
            "Preserve the original meaning, scores, checklist values, issues, and fixes. "
            "Keep strings concise. Do not add markdown fences or commentary."
        )

        repair_user = f"""
Repair the following malformed or truncated judge output into one complete valid JSON object.

Requirements:
- Keep the same top-level fields when present.
- Finish incomplete strings and arrays conservatively.
- Limit each issue/fix string to at most 35 words.
- Limit arrays to the 6 most important items.
- Return JSON only.

MALFORMED OUTPUT:
{content}
""".strip()

        repaired_data = _azure_chat_completion(
            url=AZURE_OPENAI_JUDGE_URL,
            api_key=AZURE_OPENAI_JUDGE_API_KEY,
            messages=[
                {"role": "system", "content": repair_system},
                {"role": "user", "content": repair_user},
            ],
            max_output_tokens=2400,
            use_max_completion_tokens=True,
            temperature=None,
            json_mode=True,
            request_label="GPT-5.2 judge JSON repair",
        )

        repaired_content = _extract_message_content(repaired_data)
        repaired_candidate = _extract_json_object(repaired_content)

        try:
            return json.loads(repaired_candidate)
        except json.JSONDecodeError as exc:
            raise ValueError(
                "GPT-5.2 judge failed to return valid JSON even after repair.\n"
                f"Original output preview:\n{content[:4000]}\n\n"
                f"Repair output preview:\n{repaired_content[:4000]}"
            ) from exc


def call_reviser_llm(system_prompt: str, user_prompt: str) -> str:
    """GPT-4.1 reviser."""
    data = _azure_chat_completion(
        url=AZURE_OPENAI_REVISER_URL,
        api_key=AZURE_OPENAI_REVISER_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=2400,
        use_max_completion_tokens=False,
        temperature=0.1,
        request_label="GPT-4.1 reviser",
    )
    return _extract_message_content(data)


print("Role-specific LLM helper functions ready")


## JSON-safety update

- Increased Writer and Reviser output limits to reduce incomplete sections.
- Increased GPT-5.2 Judge output allowance.
- Added automatic JSON extraction and one repair retry when the Judge returns truncated or malformed JSON.
- Judge prompts now request concise issue/fix arrays to reduce output truncation.


In [ ]:
# ── OPTIONAL ROLE ENDPOINT SMOKE TEST ────────────────────────
# Run this cell before invoking the LangGraph pipeline.
# It tests each deployment with a very small request, making it easier to
# distinguish endpoint/model problems from prompt-size or graph problems.

def test_role_endpoints() -> dict:
    results = {}

    tests = [
        ("writer_gpt_5_1", call_writer_llm, "Reply with exactly: writer ok"),
        (
            "judge_gpt_5_2",
            lambda s, u: call_judge_llm_json(s, u),
            'Return only this JSON object: {"status":"judge ok"}',
        ),
        ("reviser_gpt_4_1", call_reviser_llm, "Reply with exactly: reviser ok"),
    ]

    for name, fn, prompt in tests:
        try:
            result = fn(
                "You are performing a minimal endpoint connectivity test.",
                prompt,
            )
            results[name] = {
                "success": True,
                "response_preview": str(result)[:200],
            }
        except Exception as exc:
            results[name] = {
                "success": False,
                "error": str(exc)[:1000],
            }

    print(json.dumps(results, indent=2, ensure_ascii=False))
    return results


# Uncomment to test all three deployments before running the graphs:
# endpoint_test_results = test_role_endpoints()


In [ ]:
# ── LOAD PAYLOAD ─────────────────────────────────────────────
# Works both locally and in this sandbox if the payload is placed next to the notebook.

candidate_paths = [
    Path("Data/payload_BANK01.json"),
    Path("payload_BANK01.json"),
    Path.cwd() / "Data" / "payload_BANK01.json",
    Path.cwd() / "payload_BANK01.json",
]

PAYLOAD_PATH = next((p for p in candidate_paths if p.exists()), None)
if PAYLOAD_PATH is None:
    raise FileNotFoundError(
        "Could not find payload_BANK01.json. Put it in Data/payload_BANK01.json "
        "or in the same folder as the notebook."
    )

with open(PAYLOAD_PATH, "r", encoding="utf-8") as f:
    payload = json.load(f)

bank_name = payload["bank"]["bank_name"]
print(f"Loaded payload for: {bank_name}")
print(f"Payload path: {PAYLOAD_PATH}")


In [ ]:
# ── EVIDENCE EXTRACTOR ───────────────────────────────────────
# Pulls only the governance-relevant fields from the payload.
# Strict governance fixes added:
# - formal mandate / charter evidence is separated from activity evidence
# - board trade-off evidence is explicitly checked; if absent, the writer must state that limitation
# - management process evidence is converted into a process flow, not only an inventory
# - skills adequacy process evidence is separated from skills outcome metrics
# - assurance scope limitation is made explicit for financed emissions / Scope 3
# - meeting_id retained for board decision traceability


def _is_present(value) -> bool:
    return value is not None and str(value).strip().lower() not in {"", "nan", "none", "null"}


def _safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


def _normalise_text(value) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def extract_management_process_evidence(payload: dict, year: int = 2024) -> dict:
    """Summarise the process evidence needed for management responsibility."""
    risks = [
        r for r in payload.get("climate_risk_register", [])
        if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year
    ]

    if not risks:
        return {
            "risk_register_available": False,
            "message": "No climate_risk_register records available for the reporting year.",
            "process_flow_instruction": (
                "Do not invent management process details. State that the available evidence does not "
                "include risk-register process records for the reporting year."
            )
        }

    frequencies = sorted({str(r.get("monitoring_frequency")) for r in risks if _is_present(r.get("monitoring_frequency"))})
    risk_categories = sorted({str(r.get("risk_category")) for r in risks if _is_present(r.get("risk_category"))})
    risk_ratings = sorted({str(r.get("risk_rating")) for r in risks if _is_present(r.get("risk_rating"))})
    scenario_links = sorted({str(r.get("scenario_analysis_link")) for r in risks if _is_present(r.get("scenario_analysis_link"))})
    mitigation_actions = sorted({str(r.get("mitigation_actions")) for r in risks if _is_present(r.get("mitigation_actions"))})

    integrated_count = sum(1 for r in risks if r.get("erm_integrated_flag") is True)
    changed_count = sum(1 for r in risks if r.get("changed_since_prior_period") is True)

    # Keep only the most useful examples for prompt compactness.
    material_risk_examples = []
    rating_priority = {"critical": 4, "high": 3, "medium": 2, "low": 1}
    sorted_risks = sorted(
        risks,
        key=lambda r: (
            rating_priority.get(str(r.get("risk_rating", "")).lower(), 0),
            float(r.get("financial_impact_meur") or 0)
        ),
        reverse=True,
    )
    for r in sorted_risks[:5]:
        material_risk_examples.append({
            "risk_id": r.get("risk_id"),
            "risk_name": r.get("risk_name"),
            "risk_category": r.get("risk_category"),
            "risk_rating": r.get("risk_rating"),
            "time_horizon": r.get("time_horizon"),
            "monitoring_frequency": r.get("monitoring_frequency"),
            "erm_integrated_flag": r.get("erm_integrated_flag"),
            "scenario_analysis_link": r.get("scenario_analysis_link"),
            "mitigation_actions": r.get("mitigation_actions"),
        })

    return {
        "risk_register_available": True,
        "reporting_year": year,
        "risk_count": len(risks),
        "erm_integrated_count": integrated_count,
        "changed_since_prior_period_count": changed_count,
        "monitoring_frequencies": frequencies,
        "risk_categories": risk_categories,
        "risk_ratings": risk_ratings,
        "scenario_analysis_links": scenario_links,
        "mitigation_actions": mitigation_actions[:8],
        "material_risk_examples": material_risk_examples,
        "process_flow_instruction": (
            "Write management responsibility as a process flow: identify climate risks in the climate risk register; "
            "classify them by category, time horizon and rating; monitor them at the recorded quarterly or semi-annual "
            "frequency; link relevant risks to scenario analysis where a scenario link exists; define mitigation actions; "
            "and use ERM integration to support management monitoring and board or committee review where required. "
            "Do not invent a formal escalation threshold unless explicitly provided."
        )
    }


def extract_governance_evidence(payload: dict) -> dict:

    gov_records = payload.get("governance", [])
    board_minutes = payload.get("board_minutes", [])
    bank = payload.get("bank", {})
    reporting_kpis = payload.get("reporting_kpis", {})

    gov_by_year = {
        str(r["reporting_year"]): r
        for r in gov_records
        if isinstance(r, dict) and "reporting_year" in r
    }

    gov_trend = []
    for year in ["2022", "2023", "2024"]:
        if year in gov_by_year:
            g = gov_by_year[year]
            gov_trend.append({
                "year": int(year),
                "esg_committee_meetings": g.get("esg_committee_meetings_per_year"),
                "board_climate_expertise_pct": g.get("board_climate_expertise_pct"),
                "ceo_esg_compensation_pct": g.get("ceo_esg_compensation_pct"),
                "all_exec_climate_remuneration_pct": g.get("all_exec_climate_remuneration_pct"),
                "climate_on_board_agenda_pct": g.get("climate_on_board_agenda_pct"),
                "management_committee_name": g.get("management_committee_name"),
                "board_full_meeting_frequency": g.get("board_full_meeting_frequency"),
            })

    PRIORITY_TOPICS = [
        "scenario", "transition", "net_zero", "target", "carbon_credit",
        "remuneration", "tcfd", "green_finance", "physical_risk", "risk", "esg"
    ]

    def decision_score(m: dict) -> int:
        topics = str(m.get("climate_topics_discussed", "")).lower()
        decision = str(m.get("decision_summary", "")).lower()
        ifrs = str(m.get("ifrs_s2_para_evidence", "")).lower()
        score = sum(1 for t in PRIORITY_TOPICS if t in topics or t in decision)
        if "6(a)(v)" in ifrs:
            score += 2
        if str(m.get("committee_type", "")).lower() == "full_board":
            score += 1
        return score

    minutes_2024 = [
        m for m in board_minutes
        if isinstance(m, dict)
        and _safe_int(m.get("reporting_year")) == 2024
        and m.get("decision_made_flag") is True
        and _is_present(m.get("decision_summary"))
        and _is_present(m.get("meeting_id"))
    ]

    # Deduplicate by decision text only; keep the highest-scoring exact record and retain meeting_id.
    best_by_decision = {}
    for m in minutes_2024:
        decision_text = re.sub(r"\s+", " ", str(m.get("decision_summary", "")).lower().strip())
        current = best_by_decision.get(decision_text)
        if current is None or decision_score(m) > decision_score(current):
            best_by_decision[decision_text] = m

    selected_decisions = []
    for m in sorted(best_by_decision.values(), key=decision_score, reverse=True)[:6]:
        selected_decisions.append({
            "meeting_id": m.get("meeting_id"),
            "date": m.get("meeting_date"),
            "committee": m.get("committee_name"),
            "committee_type": m.get("committee_type"),
            "topics_discussed": m.get("climate_topics_discussed"),
            "decision": m.get("decision_summary"),
            "ifrs_evidence_para": m.get("ifrs_s2_para_evidence"),
            "internal_ref": f"[REF:{m.get('meeting_id')}]",
        })

    gov_2024 = gov_by_year.get("2024", {})

    # Evidence gap assessment for strict governance disclosure.
    # These are intentionally conservative: the writer may state a limitation, but must not invent missing details.
    governance_instrument_fields = [
        "committee_charter", "committee_terms_of_reference", "board_mandate", "esg_committee_mandate",
        "formal_climate_mandate", "governance_policy_reference", "committee_charter_climate_mandate"
    ]
    formal_mandate_available = any(_is_present(gov_2024.get(f)) for f in governance_instrument_fields)

    tradeoff_terms = ["tradeoff", "trade-off", "capital allocation", "profitability", "cost", "risk appetite", "competing"]
    tradeoff_decisions = [
        m for m in minutes_2024
        if any(term in str(m.get("decision_summary", "")).lower() or term in str(m.get("climate_topics_discussed", "")).lower() for term in tradeoff_terms)
    ]
    board_tradeoff_evidence_available = len(tradeoff_decisions) > 0

    skills_process_fields = [
        "skills_matrix", "skills_assessment_process", "board_skills_review", "skills_adequacy_assessment",
        "director_training_frequency", "training_hours", "skills_gap_analysis"
    ]
    skills_adequacy_process_available = any(_is_present(gov_2024.get(f)) for f in skills_process_fields)

    assurance_scope = str(gov_2024.get("assurance_scope", ""))
    financed_emissions_2024 = reporting_kpis.get("financed_emissions_2024_tco2e")
    assurance_scope_limitation = {
        "assurance_scope": assurance_scope,
        "external_assurance": gov_2024.get("external_assurance"),
        "provider": gov_2024.get("assurance_provider"),
        "standard": gov_2024.get("assurance_standard"),
        "financed_emissions_2024_tco2e": financed_emissions_2024,
        "financed_emissions_in_scope": "financed" in assurance_scope.lower() or "scope 3" in assurance_scope.lower(),
        "instruction": (
            "State that assurance covers only the stated scope. If the stated scope is Scope 1 and 2 emissions, "
            "do not imply financed emissions or other Scope 3 categories are assured. For a bank, explicitly clarify "
            "that financed emissions are outside the stated assurance scope based on available evidence."
        )
    }

    return {
        "bank": {
            "name": bank.get("bank_name"),
            "country": bank.get("country"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "regulatory_regime": bank.get("regulatory_regime"),
        },
        "reporting_year": 2024,
        "comparative_years": [2022, 2023],
        "governance_2024": {
            "board_size": gov_2024.get("board_size"),
            "independent_directors_pct": gov_2024.get("independent_directors_pct"),
            "esg_committee_exists": gov_2024.get("esg_committee_exists"),
            "esg_committee_meetings_per_year": gov_2024.get("esg_committee_meetings_per_year"),
            "board_climate_expertise_pct": gov_2024.get("board_climate_expertise_pct"),
            "ceo_compensation_esg_linked": gov_2024.get("ceo_compensation_esg_linked"),
            "ceo_esg_compensation_pct": gov_2024.get("ceo_esg_compensation_pct"),
            "all_exec_climate_remuneration_pct": gov_2024.get("all_exec_climate_remuneration_pct"),
            "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
            "climate_on_board_agenda_pct": gov_2024.get("climate_on_board_agenda_pct"),
            "board_full_meeting_frequency": gov_2024.get("board_full_meeting_frequency"),
            "management_committee_name": gov_2024.get("management_committee_name"),
            "erm_integration_flag": gov_2024.get("erm_integration_flag"),
            "skills_development_programme": gov_2024.get("skills_development_programme"),
            "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
            "external_assurance": gov_2024.get("external_assurance"),
            "assurance_provider": gov_2024.get("assurance_provider"),
            "assurance_scope": gov_2024.get("assurance_scope"),
            "assurance_standard": gov_2024.get("assurance_standard"),
            "tcfd_aligned": gov_2024.get("tcfd_aligned"),
            "ifrs_s2_aligned": gov_2024.get("ifrs_s2_aligned"),
        },
        "governance_trend": gov_trend,
        "management_process_evidence": extract_management_process_evidence(payload, year=2024),
        "board_decisions_2024": selected_decisions,
        "strict_governance_evidence": {
            "formal_governance_mandate_available": formal_mandate_available,
            "formal_governance_mandate_instruction": (
                "Do not claim the ESG & Sustainability Committee has a formal climate mandate unless charter/terms-of-reference evidence is provided. "
                "If no formal instrument is available, say the payload evidences committee activity and meeting frequency but does not include the committee charter or terms of reference."
            ),
            "board_tradeoff_evidence_available": board_tradeoff_evidence_available,
            "tradeoff_decisions": tradeoff_decisions[:3],
            "board_tradeoff_instruction": (
                "Discuss board trade-offs only if explicit trade-off evidence exists. If not, state that the board decision evidence identifies climate-related decisions, "
                "but does not describe specific trade-offs such as profitability, capital allocation, implementation cost, risk appetite or competing strategic priorities."
            ),
            "skills_adequacy_process_available": skills_adequacy_process_available,
            "skills_adequacy_instruction": (
                "Use the board climate expertise percentage and skills development programme as outcome/activity evidence. "
                "Do not invent a formal skills adequacy assessment process. If no skills assessment evidence exists, say the payload does not describe a formal board skills adequacy assessment process."
            ),
            "assurance_scope_limitation": assurance_scope_limitation,
        },
        "interpretation_notes": {
            "climate_on_board_agenda_pct": (
                "This figure represents the percentage of board meetings during the year "
                "where climate-related topics appeared on the agenda. It does NOT mean "
                "percentage of agenda time devoted to climate."
            ),
            "management_committee_names": (
                "Committee names are recorded by year only. The evidence does not prove that "
                "one committee evolved into, replaced, or was renamed as another. State the 2024 "
                "committee name and, if comparative names are used, present them neutrally."
            ),
            "board_decision_traceability": (
                "Each selected decision includes a meeting_id for audit traceability. Meeting IDs "
                "may be used internally but should not be printed in the final report unless required."
            ),
            "avoid_duplication": (
                "Do not list the same board decisions twice. Board oversight should summarise decision governance; "
                "the detailed dated list belongs only in the Board and committee decisions subsection."
            )
        }
    }


evidence = extract_governance_evidence(payload)

print(f"Evidence extracted for: {evidence['bank']['name']}")
print(f"Board decisions selected: {len(evidence['board_decisions_2024'])}")
print(f"Trend years: {[t['year'] for t in evidence['governance_trend']]}")
print(f"Risk-register records for management process: {evidence['management_process_evidence'].get('risk_count')}")
print("Strict governance evidence flags:")
for k, v in evidence["strict_governance_evidence"].items():
    if isinstance(v, bool):
        print(f"- {k}: {v}")
print("Selected decisions with internal refs:")
for d in evidence["board_decisions_2024"]:
    print(f"- {d['date']} | {d['committee']} | {d['decision']} | {d['internal_ref']}")


In [ ]:
# ── GOVERNANCE EVIDENCE AVAILABILITY + SAVING ────────────────
# The raw Governance payload remains the source of truth.
# The compact evidence is the agent-ready input used by Writer/Judge/Reviser.

def build_governance_availability_profile(evidence: dict) -> dict:
    gov = evidence.get("governance_2024", {})
    trend = evidence.get("governance_trend", [])
    management = evidence.get("management_process_evidence", {})
    strict = evidence.get("strict_governance_evidence", {})
    decisions = evidence.get("board_decisions_2024", [])

    def present(value) -> bool:
        return value is not None and str(value).strip().lower() not in {
            "", "none", "null", "nan"
        }

    trend_metrics = {
        "esg_committee_meetings": [
            row.get("esg_committee_meetings") for row in trend
            if present(row.get("esg_committee_meetings"))
        ],
        "board_climate_expertise_pct": [
            row.get("board_climate_expertise_pct") for row in trend
            if present(row.get("board_climate_expertise_pct"))
        ],
        "ceo_esg_compensation_pct": [
            row.get("ceo_esg_compensation_pct") for row in trend
            if present(row.get("ceo_esg_compensation_pct"))
        ],
        "all_exec_climate_remuneration_pct": [
            row.get("all_exec_climate_remuneration_pct") for row in trend
            if present(row.get("all_exec_climate_remuneration_pct"))
        ],
        "climate_on_board_agenda_pct": [
            row.get("climate_on_board_agenda_pct") for row in trend
            if present(row.get("climate_on_board_agenda_pct"))
        ],
    }

    assurance_scope = str(gov.get("assurance_scope") or "").lower()
    financed_emissions = evidence.get("assurance_context", {}).get(
        "financed_emissions_2024_tco2e"
    )

    return {
        "board_core_metrics_available": all(
            present(gov.get(field))
            for field in [
                "board_size",
                "independent_directors_pct",
                "esg_committee_meetings_per_year",
                "climate_risk_reporting_to_board",
                "climate_on_board_agenda_pct",
            ]
        ),
        "trend_metrics_available": {
            name: len(values) >= 2
            for name, values in trend_metrics.items()
        },
        "selected_decision_count": len(decisions),
        "board_decisions_available": len(decisions) > 0,
        "formal_governance_mandate_available": bool(
            strict.get("formal_governance_mandate_available")
        ),
        "board_tradeoff_evidence_available": bool(
            strict.get("board_tradeoff_evidence_available")
        ),
        "management_process_available": bool(
            management.get("risk_register_available")
        ),
        "formal_escalation_thresholds_available": bool(
            management.get("formal_escalation_thresholds_available", False)
        ),
        "skills_outcome_metrics_available": present(
            gov.get("board_climate_expertise_pct")
        ),
        "skills_development_programme_available": bool(
            gov.get("skills_development_programme")
        ),
        "skills_adequacy_process_available": bool(
            strict.get("skills_adequacy_process_available")
        ),
        "remuneration_evidence_available": (
            present(gov.get("ceo_esg_compensation_pct"))
            and present(gov.get("all_exec_climate_remuneration_pct"))
        ),
        "assurance_evidence_available": all(
            present(gov.get(field))
            for field in [
                "external_assurance",
                "assurance_provider",
                "assurance_scope",
                "assurance_standard",
            ]
        ),
        "assurance_scope_is_scope1_scope2_only": (
            "scope 1" in assurance_scope and "scope 2" in assurance_scope
        ),
        "financed_emissions_available_for_scope_context": present(
            financed_emissions
        ),
        "writer_policy": {
            "use_available_evidence": (
                "Use every material evidence item that is available and relevant."
            ),
            "handle_unavailable_evidence": (
                "When a material governance requirement is not supported by the "
                "available evidence, state the evidence boundary once in the "
                "relevant subsection. Do not invent the missing process or control."
            ),
            "wording": (
                "Use 'available evidence' or 'available documentation' in the "
                "final disclosure; do not use the technical word 'payload'."
            ),
        },
    }


def add_governance_traceability(evidence: dict, source_payload_path: Path) -> dict:
    evidence = dict(evidence)
    evidence["availability_profile"] = build_governance_availability_profile(evidence)
    evidence["source_traceability"] = {
        "source_payload_path": str(source_payload_path),
        "source_tables": [
            "bank",
            "governance",
            "board_minutes",
            "climate_risk_register",
            "reporting_kpis",
        ],
        "selected_board_decision_refs": [
            item.get("internal_ref")
            for item in evidence.get("board_decisions_2024", [])
            if item.get("internal_ref")
        ],
        "management_risk_refs": [
            item.get("risk_id")
            for item in evidence.get("management_process_evidence", {}).get(
                "material_risk_examples", []
            )
            if item.get("risk_id")
        ],
    }
    return evidence


# Enrich the compact evidence used by the agents.
evidence = add_governance_traceability(evidence, PAYLOAD_PATH)

# Make a self-contained raw Governance payload.
# climate_risk_register is included because Management Responsibility depends on it.
raw_governance_payload = {
    key: payload.get(key)
    for key in [
        "metadata",
        "bank",
        "governance",
        "board_minutes",
        "climate_risk_register",
        "reporting_kpis",
    ]
    if key in payload
}

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

RAW_GOVERNANCE_PATH = output_dir / "payload_BANK01_governance_raw.json"
COMPACT_GOVERNANCE_PATH = output_dir / "compact_governance_evidence_BANK01.json"

with open(RAW_GOVERNANCE_PATH, "w", encoding="utf-8") as f:
    json.dump(raw_governance_payload, f, indent=2, ensure_ascii=False)

with open(COMPACT_GOVERNANCE_PATH, "w", encoding="utf-8") as f:
    json.dump(evidence, f, indent=2, ensure_ascii=False)

print("Governance evidence prepared")
print(f"- Raw Governance payload: {RAW_GOVERNANCE_PATH}")
print(f"- Compact Governance evidence: {COMPACT_GOVERNANCE_PATH}")
print("- Availability profile:")
print(json.dumps(evidence["availability_profile"], indent=2, ensure_ascii=False))


In [ ]:
# ── STATE DEFINITION ─────────────────────────────────────────
class GovernanceState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict

In [ ]:
# ── GOVERNANCE REQUIREMENTS ─────────────────────────────────
# IFRS references are used internally for coverage only.
# The final markdown headings and body must NOT include IFRS paragraph references.

IFRS_GOVERNANCE_REQUIREMENTS = """
STRICT GOVERNANCE DISCLOSURE REQUIREMENTS FOR THIS SECTION:

Board oversight:
  - Describe board oversight of climate-related risks and opportunities.
  - Distinguish activity evidence from formal mandate evidence.
  - If committee charter / terms of reference / formal mandate evidence is not provided, state that the available evidence shows committee activity and meeting frequency, but does not include the formal governance instrument.
  - Explain how the board is informed about climate matters, using climate reporting cadence and board agenda evidence.
  - Explain how climate is considered in oversight of strategy, major transactions, risk management, metrics and targets.
  - Discuss board-level trade-offs only if evidenced. If no trade-off evidence exists, state that board decisions are evidenced but specific trade-offs are not described in the available minutes evidence.
  - Do not list the detailed board decisions here; summarise and point to the dedicated decisions subsection.

Management responsibility:
  - Which management body or role is responsible for climate-related risks and opportunities.
  - Write a process flow, not only an inventory: risk identification, register recording, classification, monitoring frequency, scenario links, mitigation actions, and ERM integration.
  - Be careful with escalation wording: only state a formal escalation threshold if explicit evidence exists. Otherwise use safe wording about board or committee review where required.

Climate skills and competencies:
  - Use board climate expertise and skills development programme evidence.
  - Describe the skills adequacy assessment process only if evidence exists.
  - If no skills adequacy process evidence exists, state that the payload evidences expertise percentage and skills development activity, but does not describe a formal skills adequacy assessment process.

Remuneration:
  - Explain whether and how climate-related performance metrics are incorporated into remuneration.
  - Cite percentage of CEO and all-executive remuneration linked to ESG/climate metrics where available.

Board and committee decisions:
  - Include the detailed dated list only in this subsection.
  - Use exact date, committee and decision pairings from board_decisions_2024.
  - Do not repeat the same detailed decisions in Board oversight.

External assurance and controls:
  - State assurance provider, standard, level and exact scope.
  - Explain limited assurance safely as lower assurance than reasonable assurance.
  - Do not imply assurance covers metrics outside the stated scope.
  - For a bank, if assurance covers only Scope 1 and Scope 2 emissions, clarify that financed emissions / Scope 3 are outside the stated assurance scope based on available evidence.
"""


In [ ]:
# ── WRITER SYSTEM PROMPT ─────────────────────────────────────
WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Governance section
of an IFRS S1/S2 aligned climate disclosure report for a commercial bank.

WRITING STANDARDS:
- Formal, third-person professional disclosure language suitable for publication.
- Specific and data-driven — cite exact figures, dates, and percentages.
- Every quantitative claim must come from the provided evidence — never invent numbers.
- Use IFRS requirements internally for coverage, but DO NOT put IFRS paragraph references in subsection headings or body text.
- No vague language such as "demonstrates commitment" unless backed by concrete evidence.
- Avoid overly strong assurance/control language such as "ensuring"; prefer "supporting", "providing", or "helping".
- Avoid compliance conclusions such as "aligned with IFRS S2 requirements", "fully aligned", "compliant", or "ensures reliability".
- Do not add a generic limitation disclaimer at the end.
- Specific evidence boundaries are allowed and required when evidence is missing, for example: "the available evidence does not describe...".
- Do not hedge when data is clearly available.

STRICT IFRS GOVERNANCE CONTROL:
- Distinguish board-level requirements from management-level requirements.
- Do not use management process evidence to satisfy board trade-off or board mandate disclosure.
- Board oversight must address mandate/activity, information flow, strategy/major transaction oversight, trade-offs or trade-off evidence limitation, and target/decision monitoring.
- Management responsibility must be written as a process flow, not as a raw inventory of risk register facts.
- Climate skills must separate outcome metrics from the process for assessing skills adequacy. Do not invent a skills adequacy process.
- External assurance must state the exact assurance scope and clarify when financed emissions / Scope 3 are outside that scope.
- Do not duplicate the detailed board decision list across multiple subsections.

HALLUCINATION CONTROL:
- Do not infer that a committee evolved, was renamed, replaced, strengthened, or specialized across years unless the evidence explicitly says so.
- If committee names differ by year, state only that different names are recorded across the comparative period.
- Do not create a transformation narrative from time-series values.
- Use board decision dates only from board_decisions_2024 and keep the date-decision-committee pairing exactly as provided.
- For escalation, do not write "significant or material climate risks are escalated" unless the evidence explicitly provides escalation criteria or route.
  Safer wording: "The risk register and ERM integration support management monitoring and board or committee review where required."
- For remuneration, avoid interpretive wording such as "demonstrates progressive integration". Use factual wording such as "indicates increased use".
- For assurance, explain limited assurance as lower than reasonable assurance. Do not call it "moderate assurance".

OUTPUT FORMAT:
Return markdown with exactly this subsection structure and NO IFRS paragraph references in headings or body text:

### Governance

#### Board oversight
...content...

#### Management responsibility
...content...

#### Climate skills and competencies
...content...

#### Remuneration and climate incentives
...content...

#### Board and committee decisions during 2024
...content...

#### External assurance and controls
...content...
""".strip()


def build_writer_prompt(evidence: dict, judge_feedback: str = None) -> str:
    """
    Build an evidence-aware Governance prompt.

    The prompt requires the writer to use available evidence and to state
    boundaries only where evidence is genuinely unavailable.
    """
    profile = evidence.get("availability_profile", {})
    is_revision = judge_feedback is not None

    availability_instructions = []

    if profile.get("formal_governance_mandate_available"):
        availability_instructions.append(
            "- Describe the formal committee/board mandate using the supplied direct evidence."
        )
    else:
        availability_instructions.append(
            "- Committee activity is evidenced, but a formal charter/terms of reference is not. "
            "State this boundary once in Board oversight without inventing a formal mandate."
        )

    if profile.get("board_tradeoff_evidence_available"):
        availability_instructions.append(
            "- Describe only the specific board trade-offs included in the evidence."
        )
    else:
        availability_instructions.append(
            "- Board decisions are available, but specific board trade-offs are not documented. "
            "State this boundary once; do not invent trade-offs."
        )

    if profile.get("management_process_available"):
        availability_instructions.append(
            "- Describe Management responsibility as a process flow using the risk-register summary."
        )
    else:
        availability_instructions.append(
            "- Risk-register process evidence is unavailable. Do not invent a management process; "
            "state the evidence boundary in Management responsibility."
        )

    if profile.get("formal_escalation_thresholds_available"):
        availability_instructions.append(
            "- Describe formal escalation thresholds/routes exactly as evidenced."
        )
    else:
        availability_instructions.append(
            "- Formal escalation thresholds/routes are not evidenced. Do not imply a formal escalation design."
        )

    if profile.get("skills_adequacy_process_available"):
        availability_instructions.append(
            "- Describe the formal board skills adequacy assessment process from evidence."
        )
    else:
        availability_instructions.append(
            "- Board expertise metrics and a skills programme may be available, but a formal skills "
            "adequacy assessment process is not evidenced. State that boundary without inventing one."
        )

    decision_count = profile.get("selected_decision_count", 0)
    if decision_count:
        availability_instructions.append(
            f"- Use the {decision_count} selected dated board/committee decisions in the dedicated decisions subsection."
        )
    else:
        availability_instructions.append(
            "- No selected board decisions are available. Do not invent decisions or dates."
        )

    if profile.get("assurance_evidence_available"):
        availability_instructions.append(
            "- Describe assurance using the exact provider, standard, level, and stated scope."
        )
        if profile.get("assurance_scope_is_scope1_scope2_only"):
            availability_instructions.append(
                "- Clarify that assurance is limited to Scope 1 and Scope 2. "
                "Where financed-emissions context is available, state that financed emissions/Scope 3 "
                "are outside the stated assurance scope."
            )
    else:
        availability_instructions.append(
            "- Assurance evidence is incomplete or unavailable. Do not invent assurance conclusions."
        )

    trend_availability = profile.get("trend_metrics_available", {})
    available_trends = [name for name, available in trend_availability.items() if available]
    unavailable_trends = [name for name, available in trend_availability.items() if not available]

    base_instructions = f"""
BANK: {evidence['bank']['name']} ({evidence['bank']['country']})
REPORTING YEAR: {evidence['reporting_year']}
COMPARATIVE YEARS: {evidence['comparative_years']}

REQUIRED FINAL STRUCTURE:
### Governance
#### Board oversight
#### Management responsibility
#### Climate skills and competencies
#### Remuneration and climate incentives
#### Board and committee decisions during 2024
#### External assurance and controls

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

EVIDENCE-AWARE INSTRUCTIONS:
{chr(10).join(availability_instructions)}

TREND INSTRUCTIONS:
- Use comparative trends only for metrics with at least two available years.
- Available trend metrics: {available_trends}
- Do not invent or force unavailable trends: {unavailable_trends}

COMPACT GOVERNANCE EVIDENCE — USE ONLY THIS DATA:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

GENERAL WRITING RULES:
- Write formal, publication-ready disclosure language.
- Use exact figures, dates, committee names, and decision pairings from evidence.
- Do not infer that differently named committees evolved, were renamed, replaced, or strengthened.
- Interpret climate_on_board_agenda_pct as the percentage of board meetings where climate appeared on the agenda.
- A boolean such as major_transactions_climate_check indicates evidence of climate checks; it does not by itself prove a formal policy or mandatory control requirement.
- Do not duplicate the detailed decision list in Board oversight.
- Use the terms "available evidence" or "available documentation", not "payload", in the final section.
- Never invent evidence, controls, thresholds, policies, trade-offs, assurance coverage, or assessment processes.
- Do not include visible IFRS paragraph references in the final section.
""".strip()

    if is_revision:
        return f"""
{base_instructions}

JUDGE FEEDBACK TO ADDRESS:
{judge_feedback}

REVISION RULES:
- Fix every supported issue raised by the judge.
- Do not invent missing evidence to satisfy the judge.
- Preserve correct content and the required structure.
- Return only the complete revised Governance section.
""".strip()

    return f"""
{base_instructions}

Write the complete Governance section now.
Return only the final Governance section.
""".strip()



In [ ]:
# ── JUDGE SYSTEM PROMPT ──────────────────────────────────────
JUDGE_SYSTEM = """
You are a strict IFRS S1/S2 compliance reviewer and ESG audit specialist.
Your job is to identify genuine gaps in a governance disclosure section — not to reward fluent writing.

CORE PRINCIPLE:
- A limitation statement reduces hallucination risk, but it is NOT equivalent to direct evidence.
- Do not treat "limitation disclosed" as full coverage.
- A section can be approved with limitations, but it should not receive a 9 or 10 when material IFRS governance elements are covered only by limitation statements.

SCORING ANCHOR:
  10: Audit-ready. All required governance elements are directly evidenced, specific, complete, non-duplicative, and no limitation statement is needed.
  9: Strong. Minor wording issues only. No material missing evidence and no material requirement covered only by limitation.
  8: Good but incomplete. One material governance element is addressed through a limitation statement rather than direct evidence.
  7: Usable draft with limitations. Two or more material governance elements are addressed through limitation statements, or one governance process is thin but honestly disclosed.
  6: Needs revision. Missing or weak coverage of one core IFRS governance requirement.
  5 or below: Not approved. Unsupported claims, wrong numbers, major IFRS coverage failure, or hallucination risk.

MATERIAL LIMITATION ITEMS:
These should be tracked separately as evidence vs limitation:
- formal governance mandate / committee charter or terms of reference
- board consideration of climate-related trade-offs
- formal board skills adequacy assessment process
- formal escalation thresholds for climate risks
- assurance scope excludes financed emissions / Scope 3 where those are material for a bank

A score of 9 or 10 requires ALL of the following to be true:
- All 6 subsections present with substantive content.
- No visible IFRS paragraph references appear in subsection headings or body text.
- climate_on_board_agenda_pct correctly interpreted as meeting frequency, not agenda time.
- At least 4 distinct board/committee decisions cited with exact dates in the dedicated decisions subsection.
- Detailed board decisions are not duplicated in the Board oversight subsection.
- Year-on-year trends present for board expertise, CEO compensation, ESG committee meetings and climate agenda frequency.
- Both CEO ESG % and all-executive climate % cited in remuneration.
- Formal governance mandate is directly evidenced, not merely disclosed as unavailable.
- Board trade-offs are directly evidenced, not merely disclosed as unavailable.
- Management responsibility is written as a process flow: risk identification/register, classification, monitoring frequency, scenario links, mitigation actions and ERM integration.
- Escalation thresholds or route are directly evidenced if claimed; otherwise a limitation is stated.
- No unsupported committee evolution / renaming / strengthening narrative.
- Skills adequacy process is directly evidenced, not merely disclosed as unavailable.
- Limited assurance is explained safely as lower assurance than reasonable assurance.
- Assurance scope limitation is clearly disclosed, especially that financed emissions / Scope 3 are outside the stated assurance scope when applicable.
- No generic limitation disclaimer at the end.
- No unsupported claims such as "ensures", "guarantees", "fully aligned", "fully resilient", "compliant", or "aligned with IFRS S2 requirements".

SCORE CAPS:
- If any required subsection is missing: maximum score 6.
- If any unsupported strong claim or hallucination appears: maximum score 6.
- If detailed decisions are duplicated: maximum score 7.
- If management is only an inventory and not a process flow: maximum score 7.
- If one material limitation item is present: maximum score 8.
- If two or three material limitation items are present: maximum score 8.
- If four or more material limitation items are present: maximum score 7.
- If the output is evidence-based but has multiple disclosed limitations, it should usually be "approved_with_limitations", not fully approved.

You must return valid JSON only — no other text.
""".strip()


def build_judge_prompt(draft: str, evidence: dict) -> str:
    """
    Build an availability-aware judge prompt.

    The judge must distinguish:
    - evidence that was available but omitted/misstated;
    - evidence that was unavailable and correctly disclosed as a boundary;
    - unsupported claims invented by the writer.
    """
    profile = evidence.get("availability_profile", {})

    return f"""
Evaluate the Governance draft against the compact evidence and the availability profile.

DRAFT:
{draft}

AVAILABILITY PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

COMPACT GOVERNANCE EVIDENCE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

EVALUATION PRINCIPLES:
1. Penalise a claim when it contradicts the evidence, invents a process/control, or overstates what a boolean/metric proves.
2. Penalise omission when material evidence is available but not used.
3. Do not demand unavailable evidence from the writer.
4. When material evidence is unavailable, an accurate and concise evidence-boundary statement is appropriate, but it lowers completeness.
5. Distinguish activity evidence from formal mandate/policy evidence.
6. Distinguish management process evidence from board-level mandate and trade-off evidence.
7. Verify dates, figures, trends, committee names, decisions, assurance scope, and all strong claims.
8. Check that the final section contains no visible IFRS paragraph references.
9. Check that detailed board decisions appear only in the dedicated decisions subsection.
10. Do not reward fluent wording when evidence use is incomplete or inaccurate.

SCORING:
- 9–10: materially complete and directly evidenced; no material evidence boundary required.
- 8: strong but one material item is unavailable or one minor evidence-use issue remains.
- 7: usable with multiple correctly disclosed evidence boundaries.
- 6: revision required because of unsupported claims, omitted available evidence, or weak coverage.
- 5 or below: major factual/evidence failures.

Return valid JSON only. Keep all issue/fix strings concise (maximum 35 words each) and include no more than 6 items per array:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approval_status": "<approved|approved_with_limitations|revision_required|rejected>",
  "approved": <true if approved or approved_with_limitations, otherwise false>,
  "checklist": {{
    "required_structure_present": <true/false>,
    "available_board_metrics_used_correctly": <true/false>,
    "available_trends_used_correctly": <true/false>,
    "available_decisions_used_correctly": <true/false>,
    "formal_mandate_handled_according_to_availability": <true/false>,
    "board_tradeoffs_handled_according_to_availability": <true/false>,
    "management_process_handled_according_to_availability": <true/false>,
    "escalation_handled_according_to_availability": <true/false>,
    "skills_handled_according_to_availability": <true/false>,
    "remuneration_evidence_used_correctly": <true/false>,
    "assurance_scope_used_correctly": <true/false>,
    "no_unsupported_committee_evolution": <true/false>,
    "no_unsupported_strong_claims": <true/false>,
    "no_visible_ifrs_refs": <true/false>,
    "no_duplicate_decisions": <true/false>
  }},
  "available_evidence_omitted": [<specific available evidence omitted from the draft>],
  "unsupported_claims": [<specific unsupported claims>],
  "correctly_disclosed_evidence_boundaries": [<accurate boundary statements>],
  "main_issues": [<specific issues>],
  "required_fixes": [<actionable evidence-aware fixes>]
}}
""".strip()



In [ ]:
# ── GOVERNANCE EVALUATION MODE ─────────────────────────────
# Deterministic rule checks and score caps have been removed.
# GPT-5.2 is solely responsible for evaluating, scoring, and approving
# the Governance section against the evidence and judge prompt.

print("Governance evaluation mode: GPT-5.2 judge only")


In [ ]:
# ── LANGGRAPH NODES ──────────────────────────────────────────

def writer_node(state: GovernanceState) -> GovernanceState:
    is_revision = state["revision_count"] > 0
    feedback = None

    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n" +
            "\n".join(f"- {fix}" for fix in issues) +
            "\n\nFAILED CHECKLIST ITEMS:\n" +
            "\n".join(f"- {item}" for item in false_items)
        )

    prompt = build_writer_prompt(state["evidence"], judge_feedback=feedback)

    draft = call_writer_llm(
        system_prompt=WRITER_SYSTEM,
        user_prompt=prompt,
    )

    print(f"\n{'='*50}")
    print(f"WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")

    return {
        **state,
        "draft": draft.strip(),
        "status": "judging"
    }



# ── STRICT SCORE CAPS ────────────────────────────────────────
def strict_governance_score(raw_score: int, checklist: dict) -> tuple[int, str, int]:
    """
    Convert a potentially generous LLM score into a stricter audit-style score.
    Limitation statements are good for honesty, but they reduce completeness.
    Returns: (final_score, cap_reason, material_limitation_count)
    """
    score = int(raw_score or 0)

    limitation_keys = [
        "formal_mandate_limitation_used",
        "board_tradeoffs_limitation_used",
        "escalation_threshold_limitation_used",
        "skills_adequacy_limitation_used",
        "financed_emissions_outside_assurance_scope",
    ]
    material_limitation_count = sum(bool(checklist.get(k, False)) for k in limitation_keys)

    cap_reasons = []

    # Missing direct evidence but disclosed as limitation still caps score.
    if material_limitation_count >= 4:
        if score > 7:
            cap_reasons.append(f"Capped at 7 because {material_limitation_count} material requirements are addressed through limitations rather than direct evidence")
        score = min(score, 7)
    elif material_limitation_count >= 1:
        if score > 8:
            cap_reasons.append(f"Capped at 8 because {material_limitation_count} material requirement(s) are addressed through limitations rather than direct evidence")
        score = min(score, 8)

    # Hard caps for actual quality failures.
    hard_failure_caps = [
        (not checklist.get("all_six_subsections_present", True), 6, "required subsection missing"),
        (not checklist.get("no_unsupported_strong_claims", True), 6, "unsupported strong claim present"),
        (not checklist.get("no_unsupported_committee_evolution", True), 6, "unsupported committee evolution/renaming narrative present"),
        (not checklist.get("no_duplicate_decisions", True), 7, "board decisions duplicated across subsections"),
        (not checklist.get("management_process_flow", True), 7, "management section is not process-based"),
        (not checklist.get("assurance_explained_safely", True), 7, "limited assurance wording is unsafe or overstated"),
        (not checklist.get("no_visible_ifrs_refs", True), 7, "visible IFRS paragraph references appear in final text"),
        (not checklist.get("assurance_scope_limitation", True), 7, "assurance scope limitation missing"),
    ]

    for condition, cap, reason in hard_failure_caps:
        if condition:
            if score > cap:
                cap_reasons.append(f"Capped at {cap}: {reason}")
            score = min(score, cap)

    return score, "; ".join(cap_reasons) if cap_reasons else "No cap applied", material_limitation_count


def derive_approval_status(score: int, checklist: dict, false_count: int, material_limitation_count: int) -> str:
    """More nuanced than approved/rejected."""
    hard_fail = (
        false_count > 0 and not (
            # limitation flags are not hard failures if handled honestly
            checklist.get("formal_mandate_limitation_used", False) or
            checklist.get("board_tradeoffs_limitation_used", False) or
            checklist.get("skills_adequacy_limitation_used", False) or
            checklist.get("escalation_threshold_limitation_used", False) or
            checklist.get("financed_emissions_outside_assurance_scope", False)
        )
    )

    if score >= 9 and material_limitation_count == 0 and false_count == 0:
        return "approved"
    if score >= 7 and not hard_fail:
        return "approved_with_limitations"
    if score >= 5:
        return "revision_required"
    return "rejected"


def judge_node(state: GovernanceState) -> GovernanceState:
    """
    Evaluate Governance using only the GPT-5.2 judge.

    No deterministic rule checks, hard-fail overrides, score caps, or
    programmatic approval changes are applied.
    """
    draft = state["draft"]

    judge_prompt = build_judge_prompt(draft, state["evidence"])
    judge_result = call_judge_llm_json(
        system_prompt=JUDGE_SYSTEM,
        user_prompt=judge_prompt,
    )

    # Preserve the judge's own decision. Add only safe defaults when omitted.
    judge_result.setdefault("approved", False)
    judge_result.setdefault(
        "approval_status",
        "approved" if judge_result.get("approved") else "revision_required",
    )
    judge_result.setdefault("main_issues", [])
    judge_result.setdefault("required_fixes", [])
    judge_result.setdefault("checklist", {})

    print("\nGOVERNANCE JUDGE RESULT — GPT-5.2 ONLY")
    print(json.dumps(judge_result, indent=2, ensure_ascii=False))

    return {
        **state,
        "judge_result": judge_result,
        "status": "judging",
    }


GOVERNANCE_REVISER_SYSTEM = """
You are a precise sustainability disclosure reviser.

Revise the existing Governance section using only:
- the supplied evidence;
- the strict governance requirements; and
- the judge's required fixes.

Rules:
- Fix every judge issue and failed checklist item.
- Preserve correct content that was not criticised.
- Never invent evidence.
- Keep the exact six-subsection structure.
- Do not add visible IFRS paragraph references.
- Return only the complete revised Governance section.
""".strip()


def reviser_node(state: GovernanceState) -> GovernanceState:
    if state["revision_count"] >= state["max_revisions"]:
        return {**state, "status": "failed", "final_section": state["draft"]}

    judge = state.get("judge_result", {})
    issues = judge.get("required_fixes", [])
    checklist = judge.get("checklist", {})
    false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]

    revision_prompt = f"""
STRICT GOVERNANCE REQUIREMENTS:
{IFRS_GOVERNANCE_REQUIREMENTS}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(state["evidence"].get("availability_profile", {}), indent=2, ensure_ascii=False)}

COMPACT GOVERNANCE EVIDENCE:
{json.dumps(state["evidence"], indent=2, ensure_ascii=False)}

REVISION BOUNDARY:
- Use available evidence when the judge identifies an omission.
- When evidence is unavailable, preserve or improve the accurate boundary statement.
- Never invent a missing process, policy, threshold, trade-off, or assurance scope.

CURRENT DRAFT:
{state["draft"]}

JUDGE REQUIRED FIXES:
{json.dumps(issues, indent=2, ensure_ascii=False)}

FAILED CHECKLIST ITEMS:
{json.dumps(false_items, indent=2, ensure_ascii=False)}

Revise the current draft and return only the complete revised Governance section.
""".strip()

    revised_draft = call_reviser_llm(
        system_prompt=GOVERNANCE_REVISER_SYSTEM,
        user_prompt=revision_prompt,
    )

    new_revision_count = state["revision_count"] + 1
    print(f"\nGovernance revised with GPT-4.1 | revision {new_revision_count}")

    return {
        **state,
        "draft": revised_draft.strip(),
        "revision_count": new_revision_count,
        "status": "judging",
    }


def finalize_node(state: GovernanceState) -> GovernanceState:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)

    print(f"\n{'='*50}")
    print(f"FINALIZED")
    print(f"  Status: {'APPROVED' if approved else 'MAX REVISIONS REACHED'}")
    print(f"  Final score: {judge.get('overall_score')}/10")
    print(f"  Revisions: {state['revision_count']}")
    print(f"{'='*50}")

    return {
        **state,
        "final_section": state["draft"],
        "status": "approved" if approved else "failed"
    }


# ── ROUTING ──────────────────────────────────────────────────
def route_after_judge(state: GovernanceState) -> str:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)
    revision_count = state.get("revision_count", 0)
    max_revisions = state.get("max_revisions", 2)

    if approved:
        return "finalize"
    if revision_count >= max_revisions:
        return "finalize"
    return "revise"

In [ ]:
# ── BUILD AND COMPILE GRAPH ───────────────────────────────────
builder = StateGraph(GovernanceState)

builder.add_node("writer",   writer_node)
builder.add_node("judge",    judge_node)
builder.add_node("reviser",  reviser_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START,      "writer")
builder.add_edge("writer",   "judge")
builder.add_edge("reviser",  "judge")
builder.add_edge("finalize", END)

builder.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "revise":   "reviser",
        "finalize": "finalize",
    }
)

graph = builder.compile()
print("Graph compiled")

In [ ]:
# ── RUN ──────────────────────────────────────────────────────
initial_state: GovernanceState = {
    "bank_name":      bank_name,
    "evidence":       evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions":  2,
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {}
}

print(f"Starting governance generation for: {bank_name}\n")
result = graph.invoke(initial_state)

In [ ]:
# ── OUTPUT ───────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL JUDGE RESULT")
print("="*60)
print(json.dumps(result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("GOVERNANCE SECTION")
print("="*60)
print(result["final_section"])

# Save outputs
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "governance_BANK01.md", "w", encoding="utf-8") as f:
    f.write(result["final_section"])

with open(output_dir / "governance_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "governance",
        "status":          result["status"],
        "approval_status": result["judge_result"].get("approval_status"),
        "raw_score":       result["judge_result"].get("raw_score_before_caps"),
        "final_score":     result["judge_result"].get("overall_score"),
        "score_cap_reason": result["judge_result"].get("score_cap_reason"),
        "revisions":       result["revision_count"],
        "approved":        result["judge_result"].get("approved"),
        "checklist":       result["judge_result"].get("checklist"),
        "issues":         result["judge_result"].get("main_issues"),
        "available_evidence_omitted": result["judge_result"].get("available_evidence_omitted"),
        "unsupported_claims": result["judge_result"].get("unsupported_claims"),
        "correctly_disclosed_evidence_boundaries": result["judge_result"].get("correctly_disclosed_evidence_boundaries"),
        "raw_governance_payload_path": str(RAW_GOVERNANCE_PATH),
        "compact_governance_evidence_path": str(COMPACT_GOVERNANCE_PATH),
    }, f, indent=2, ensure_ascii=False)

print(f"\nSaved to outputs/governance_BANK01.md")
print(f"Raw Governance payload saved to: {RAW_GOVERNANCE_PATH}")
print(f"Compact Governance evidence saved to: {COMPACT_GOVERNANCE_PATH}")


In [ ]:

# ============================================================
# STRATEGY SECTION GENERATOR
# IFRS S1/S2 strategy | uses same Azure REST helper functions
# ============================================================
# This section is added after Governance and reuses:
# - payload
# - bank_name
# - call_writer_llm()
# - call_judge_llm_json()
# - call_reviser_llm()
# - _is_present(), _safe_int(), _normalise_text()


def _safe_float(value, default=0.0):
    try:
        if value is None:
            return default
        # handle NaN from JSON payloads
        if isinstance(value, float) and value != value:
            return default
        return float(value)
    except Exception:
        return default


def _json_clean(obj):
    """Make dictionaries/lists safe for JSON prompt dumps by replacing NaN with None."""
    if isinstance(obj, dict):
        return {k: _json_clean(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_json_clean(v) for v in obj]
    if isinstance(obj, float) and obj != obj:
        return None
    return obj


def _pick_latest(records: list[dict], year: int = 2024) -> dict:
    for r in records:
        if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year:
            return r
    return records[-1] if records else {}


print("Strategy helper functions ready")


In [ ]:
# ── FAST STRATEGY EVIDENCE EXTRACTOR ────────────────────────
# This version intentionally summarizes the strategy payload before sending it to the LLM.
# It avoids passing all scenario rows, risk rows, and value-chain rows in full, which made
# the Strategy generation very slow.


def _load_strategy_payload_if_available(default_payload: dict) -> dict:
    """Use an enhanced strategy-only payload if present; otherwise reuse the main payload."""
    candidate_paths = [
        Path("Data/payload_BANK01_strategy_enhanced.json"),
        Path("payload_BANK01_strategy_enhanced.json"),
        Path("Data/payload_BANK01_strategy.json"),
        Path("payload_BANK01_strategy.json"),
        Path.cwd() / "Data" / "payload_BANK01_strategy_enhanced.json",
        Path.cwd() / "payload_BANK01_strategy_enhanced.json",
        Path.cwd() / "Data" / "payload_BANK01_strategy.json",
        Path.cwd() / "payload_BANK01_strategy.json",
    ]
    strategy_path = next((p for p in candidate_paths if p.exists()), None)
    if strategy_path:
        with open(strategy_path, "r", encoding="utf-8") as f:
            print(f"Loaded strategy-specific payload from: {strategy_path}")
            return json.load(f)
    print("No strategy-specific payload found; using main payload for strategy.")
    return default_payload


def _present(v) -> bool:
    if v is None:
        return False
    if isinstance(v, str):
        return v.strip() != "" and v.strip().lower() not in {"nan", "none", "null"}
    try:
        # Handles pandas/numpy NaN without importing pandas.
        return not (isinstance(v, float) and v != v)
    except Exception:
        return True


def _num(v):
    try:
        if not _present(v):
            return None
        return float(v)
    except Exception:
        return None


def _compact_clean(obj):
    """Remove None/NaN/empty values recursively to keep the prompt compact."""
    if isinstance(obj, dict):
        cleaned = {k: _compact_clean(v) for k, v in obj.items() if _present(v)}
        return {k: v for k, v in cleaned.items() if v not in ({}, [], None)}
    if isinstance(obj, list):
        cleaned = [_compact_clean(x) for x in obj if _present(x)]
        return [x for x in cleaned if x not in ({}, [], None)]
    if isinstance(obj, float) and obj != obj:
        return None
    return obj


def summarize_strategy_evidence(payload: dict) -> dict:
    """
    Build compact strategy evidence for the writer/judge.
    The goal is speed: pass summaries, extrema and top examples instead of raw tables.
    """
    metadata = payload.get("metadata", {})
    bank = payload.get("bank", {})
    reporting_kpis = payload.get("reporting_kpis", {})
    scenarios = [s for s in payload.get("climate_scenarios", []) if isinstance(s, dict)]
    risks_all = [r for r in payload.get("climate_risk_register", []) if isinstance(r, dict)]
    risks_2024 = [r for r in risks_all if r.get("reporting_year") == 2024]
    value_chain = [v for v in payload.get("value_chain_map", []) if isinstance(v, dict)]
    opportunities = [o for o in payload.get("climate_opportunities", []) if isinstance(o, dict)]
    financial_summary = [f for f in payload.get("financial_summary", []) if isinstance(f, dict)]
    latest_financial = next((f for f in financial_summary if f.get("reporting_year") == 2024), {})

    def max_by(rows, field):
        valid = [r for r in rows if _num(r.get(field)) is not None]
        if not valid:
            return None
        row = max(valid, key=lambda x: _num(x.get(field)))
        return {
            "scenario_id": row.get("scenario_id"),
            "scenario_name": row.get("scenario_name"),
            "scenario_type": row.get("scenario_type"),
            "horizon": row.get("horizon"),
            "horizon_year": row.get("horizon_year"),
            field: row.get(field),
        }

    # Scenario summary instead of 18 full scenario rows.
    scenario_summary = {
        "available": bool(scenarios),
        "scenario_count": len(scenarios),
        "frameworks": sorted({s.get("framework") for s in scenarios if _present(s.get("framework"))}),
        "scenario_types": sorted({s.get("scenario_type") for s in scenarios if _present(s.get("scenario_type"))}),
        "scenario_names": sorted({s.get("scenario_name") for s in scenarios if _present(s.get("scenario_name"))}),
        "horizon_years": sorted({s.get("horizon_year") for s in scenarios if _present(s.get("horizon_year"))}),
        "max_physical_risk_loss_pct_capital": max_by(scenarios, "physical_risk_loss_pct_capital"),
        "max_transition_risk_loss_pct_capital": max_by(scenarios, "transition_risk_loss_pct_capital"),
        "max_stranded_assets_estimate_meur": max_by(scenarios, "stranded_assets_estimate_meur"),
        "max_revenue_at_risk_meur": max_by(scenarios, "revenue_at_risk_meur"),
        "methodology_summary": [
            "NGFS v4 scenarios applied to banking book exposures across the available jurisdictions.",
            "Outputs are modelled estimates and must not be described as actual losses.",
            "Scenario results vary by transition pathway, physical risk assumptions, carbon price, technology readiness and time horizon.",
        ],
        "resilience_boundary": "Describe resilience only within scenario assumptions; do not claim general bank-wide resilience.",
    }

    # Keep only one representative resilience note per scenario type to avoid repeating long text.
    resilience_by_type = {}
    for s in scenarios:
        st = s.get("scenario_type")
        ra = s.get("resilience_assessment")
        if _present(st) and _present(ra) and st not in resilience_by_type:
            resilience_by_type[st] = ra
    scenario_summary["resilience_assessment_by_scenario_type"] = resilience_by_type

    # Risk summary instead of full risk register.
    physical_risks = [r for r in risks_2024 if str(r.get("risk_category", "")).startswith("physical")]
    transition_risks = [r for r in risks_2024 if str(r.get("risk_category", "")).startswith("transition")]
    top_risks = sorted(
        [
            {
                "risk_name": r.get("risk_name"),
                "risk_category": r.get("risk_category"),
                "time_horizon": r.get("time_horizon"),
                "risk_rating": r.get("risk_rating"),
                "financial_impact_meur": r.get("financial_impact_meur"),
                "mitigation_actions": r.get("mitigation_actions"),
                "scenario_analysis_link": r.get("scenario_analysis_link"),
            }
            for r in risks_2024
            if _num(r.get("financial_impact_meur")) is not None
        ],
        key=lambda x: _num(x.get("financial_impact_meur")) or 0,
        reverse=True,
    )[:5]

    risk_summary = {
        "risk_count_2024": len(risks_2024),
        "physical_risk_count": len(physical_risks),
        "transition_risk_count": len(transition_risks),
        "risk_categories": sorted({r.get("risk_category") for r in risks_2024 if _present(r.get("risk_category"))}),
        "time_horizons": sorted({r.get("time_horizon") for r in risks_2024 if _present(r.get("time_horizon"))}),
        "risk_ratings": sorted({r.get("risk_rating") for r in risks_2024 if _present(r.get("risk_rating"))}),
        "top_risks_by_financial_impact": top_risks,
    }

    # Value-chain summary; keep material examples and largest quantified exposures only.
    material_nodes = [v for v in value_chain if v.get("materiality_flag") is True]
    quantified_nodes = [v for v in value_chain if _num(v.get("financial_exposure_meur")) is not None]
    value_chain_summary = payload.get("value_chain_summary") or {
        "available": bool(value_chain),
        "total_nodes": len(value_chain),
        "material_nodes": len(material_nodes),
        "node_types": sorted({v.get("node_type") for v in value_chain if _present(v.get("node_type"))}),
        "material_value_chain_examples": [
            {
                "node_name": v.get("node_name"),
                "node_type": v.get("node_type"),
                "upstream_downstream": v.get("upstream_downstream"),
                "climate_exposure_type": v.get("climate_exposure_type"),
                "financial_exposure_meur": v.get("financial_exposure_meur"),
                "scope3_category": v.get("scope3_category"),
            }
            for v in material_nodes[:8]
        ],
        "largest_quantified_nodes": sorted(
            [
                {
                    "node_name": v.get("node_name"),
                    "node_type": v.get("node_type"),
                    "financial_exposure_meur": v.get("financial_exposure_meur"),
                    "scope3_category": v.get("scope3_category"),
                }
                for v in quantified_nodes
            ],
            key=lambda x: _num(x.get("financial_exposure_meur")) or 0,
            reverse=True,
        )[:5],
        "boundary": "Some own-operations and supplier nodes may be mapped qualitatively where financial exposure is unavailable.",
    }

    # Opportunity summary; keep estimates but force cautious wording.
    opportunity_summary = payload.get("opportunity_summary") or {
        "available": bool(opportunities),
        "opportunity_count": len(opportunities),
        "examples": [
            {
                "opportunity_type": o.get("opportunity_type"),
                "category": o.get("category"),
                "estimated_revenue_impact_meur": o.get("estimated_revenue_impact_meur"),
                "time_horizon": o.get("time_horizon"),
                "confidence_level": o.get("confidence_level"),
                "description": o.get("description"),
            }
            for o in opportunities[:6]
        ],
        "total_estimated_revenue_impact_meur": round(sum((_num(o.get("estimated_revenue_impact_meur")) or 0) for o in opportunities), 2),
        "revenue_impact_boundary": "Opportunity values are estimates and should not be described as guaranteed revenue.",
    }

    portfolio_summary = payload.get("portfolio_exposure_summary") or {
        "green_loans_meur": latest_financial.get("green_loans_meur"),
        "green_loans_pct": latest_financial.get("green_loans_pct") or reporting_kpis.get("green_loans_pct_2024"),
        "climate_capex_meur": latest_financial.get("climate_capex_meur") or reporting_kpis.get("climate_capex_2024_meur"),
        "climate_opex_meur": latest_financial.get("climate_opex_meur") or reporting_kpis.get("climate_opex_2024_meur"),
        "financed_emissions_tco2e": latest_financial.get("financed_em_loans_tco2e") or reporting_kpis.get("financed_emissions_2024_tco2e"),
        "carbon_intensity_tco2e_per_meur": latest_financial.get("carbon_intensity_tco2e_per_meur_lending") or reporting_kpis.get("carbon_intensity_2024_tco2e_per_meur"),
        "high_carbon_sector_exposure_pct": reporting_kpis.get("high_carbon_sector_exposure_pct"),
        "high_carbon_sector_exposure_meur": reporting_kpis.get("high_carbon_sector_exposure_meur"),
        "fossil_fuel_exposure_pct": reporting_kpis.get("fossil_fuel_exposure_pct"),
        "fossil_fuel_exposure_meur": reporting_kpis.get("fossil_fuel_exposure_meur"),
    }

    compact = {
        "bank": {
            "bank_id": bank.get("bank_id"),
            "bank_name": bank.get("bank_name"),
            "country": bank.get("country"),
            "reporting_year": metadata.get("reporting_year", 2024),
            "reporting_currency": bank.get("reporting_currency"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "total_loans_meur": bank.get("total_loans_meur"),
        },
        "risk_summary": risk_summary,
        "scenario_summary": scenario_summary,
        "value_chain_summary": value_chain_summary,
        "opportunity_summary": opportunity_summary,
        "portfolio_summary": portfolio_summary,
        "targets": reporting_kpis.get("target_summary", []),
        "business_model_impacts": payload.get("business_model_impacts", []),
        "strategy_tradeoff_decisions": payload.get("strategy_tradeoff_decisions", []),
        "evidence_boundaries": {
            "scenario_outputs_are_modelled_estimates": True,
            "opportunity_impacts_are_estimates": True,
            "do_not_overclaim_resilience": True,
            "do_not_overclaim_paris_alignment": True,
            "business_model_detail_available": bool(payload.get("business_model_impacts")),
            "value_chain_detail_available": bool(value_chain_summary),
            "tradeoff_evidence_available": bool(payload.get("strategy_tradeoff_decisions")),
            "high_carbon_and_fossil_exposure_available": _present(portfolio_summary.get("high_carbon_sector_exposure_pct")) and _present(portfolio_summary.get("fossil_fuel_exposure_pct")),
        },
    }
    return _compact_clean(compact)


strategy_payload = _load_strategy_payload_if_available(payload)
strategy_evidence = summarize_strategy_evidence(strategy_payload)

raw_size = len(json.dumps(strategy_payload, ensure_ascii=False))
compact_size = len(json.dumps(strategy_evidence, ensure_ascii=False))
print("Compact Strategy evidence ready")
print(f"Raw strategy payload size: {raw_size:,} chars")
print(f"Compact evidence size: {compact_size:,} chars")
print(f"Reduction: {round((1 - compact_size / max(raw_size, 1)) * 100, 1)}%")
print(json.dumps({
    "bank": strategy_evidence["bank"],
    "scenario_count": strategy_evidence["scenario_summary"].get("scenario_count"),
    "risk_count_2024": strategy_evidence["risk_summary"].get("risk_count_2024"),
    "value_chain_available": strategy_evidence["value_chain_summary"].get("available", True),
    "opportunities_available": strategy_evidence["opportunity_summary"].get("available", True),
    "business_model_impacts": len(strategy_evidence.get("business_model_impacts", [])),
    "strategy_tradeoffs": len(strategy_evidence.get("strategy_tradeoff_decisions", [])),
    "high_carbon_pct": strategy_evidence["portfolio_summary"].get("high_carbon_sector_exposure_pct"),
    "fossil_fuel_pct": strategy_evidence["portfolio_summary"].get("fossil_fuel_exposure_pct"),
}, indent=2, ensure_ascii=False))


In [ ]:
# ── STRATEGY EVIDENCE AVAILABILITY + SAVING ──────────────────
# The raw Strategy payload remains the source of truth.
# The compact Strategy evidence is the agent-ready input used by Writer/Judge/Reviser.

def build_strategy_availability_profile(evidence: dict) -> dict:
    risk = evidence.get("risk_summary", {})
    scenarios = evidence.get("scenario_summary", {})
    value_chain = evidence.get("value_chain_summary", {})
    opportunities = evidence.get("opportunity_summary", {})
    portfolio = evidence.get("portfolio_summary", {})
    boundaries = evidence.get("evidence_boundaries", {})
    targets = evidence.get("targets", [])
    business_model_impacts = evidence.get("business_model_impacts", [])
    tradeoffs = evidence.get("strategy_tradeoff_decisions", [])

    def present(value) -> bool:
        return value is not None and str(value).strip().lower() not in {
            "", "none", "null", "nan"
        }

    scenario_assumptions_available = any(
        present(scenarios.get(field))
        for field in [
            "frameworks",
            "scenario_types",
            "scenario_names",
            "horizon_years",
            "methodology_summary",
        ]
    )

    resilience_evidence_available = bool(
        scenarios.get("resilience_assessment_by_scenario_type")
    )

    paris_flag_or_framework_available = (
        any(present(t.get("framework")) for t in targets if isinstance(t, dict))
        or any(
            "paris" in str(value).lower()
            for value in scenarios.values()
            if present(value)
        )
    )

    quantified_value_chain_available = bool(
        value_chain.get("largest_quantified_nodes")
        or any(
            present(item.get("financial_exposure_meur"))
            for item in value_chain.get("material_value_chain_examples", [])
            if isinstance(item, dict)
        )
    )

    opportunity_estimates_available = any(
        present(item.get("estimated_revenue_impact_meur"))
        for item in opportunities.get("examples", [])
        if isinstance(item, dict)
    ) or present(opportunities.get("total_estimated_revenue_impact_meur"))

    return {
        "physical_risk_evidence_available": bool(risk.get("physical_risk_count")),
        "transition_risk_evidence_available": bool(risk.get("transition_risk_count")),
        "time_horizons_available": bool(risk.get("time_horizons"))
        or bool(scenarios.get("horizon_years")),
        "business_model_detail_available": bool(business_model_impacts),
        "value_chain_detail_available": bool(value_chain),
        "quantified_value_chain_exposure_available": quantified_value_chain_available,
        "strategy_tradeoff_evidence_available": bool(tradeoffs),
        "portfolio_financial_metrics_available": any(
            present(portfolio.get(field))
            for field in [
                "green_loans_meur",
                "green_loans_pct",
                "financed_emissions_tco2e",
                "carbon_intensity_tco2e_per_meur",
            ]
        ),
        "resource_allocation_evidence_available": (
            present(portfolio.get("climate_capex_meur"))
            or present(portfolio.get("climate_opex_meur"))
        ),
        "high_carbon_exposure_available": (
            present(portfolio.get("high_carbon_sector_exposure_pct"))
            or present(portfolio.get("high_carbon_sector_exposure_meur"))
        ),
        "fossil_fuel_exposure_available": (
            present(portfolio.get("fossil_fuel_exposure_pct"))
            or present(portfolio.get("fossil_fuel_exposure_meur"))
        ),
        "climate_opportunities_available": bool(opportunities.get("examples")),
        "quantified_opportunity_estimates_available": opportunity_estimates_available,
        "scenario_analysis_available": bool(scenarios.get("available")),
        "scenario_assumptions_available": scenario_assumptions_available,
        "scenario_financial_outputs_available": any(
            scenarios.get(field)
            for field in [
                "max_physical_risk_loss_pct_capital",
                "max_transition_risk_loss_pct_capital",
                "max_stranded_assets_estimate_meur",
                "max_revenue_at_risk_meur",
            ]
        ),
        "resilience_evidence_available": resilience_evidence_available,
        "paris_flag_or_framework_available": paris_flag_or_framework_available,
        "scenario_outputs_are_modelled_estimates": bool(
            boundaries.get("scenario_outputs_are_modelled_estimates", True)
        ),
        "opportunity_impacts_are_estimates": bool(
            boundaries.get("opportunity_impacts_are_estimates", True)
        ),
        "writer_policy": {
            "use_available_evidence": (
                "Use every material Strategy evidence item that is available and relevant."
            ),
            "handle_unavailable_evidence": (
                "When a material Strategy disclosure is unsupported by available evidence, "
                "state the boundary once in the relevant subsection and do not invent it."
            ),
            "scenario_wording": (
                "Describe scenario financial outputs as modelled estimates, not actual losses."
            ),
            "opportunity_wording": (
                "Describe opportunity revenue impacts as estimates with their confidence level, "
                "not guaranteed future revenue."
            ),
            "resilience_wording": (
                "Confine resilience statements to the relevant scenario assumptions and boundaries."
            ),
            "final_language": (
                "Use 'available evidence', 'available documentation', or 'source data'; "
                "do not use the technical word 'payload' in the final disclosure."
            ),
        },
    }


def add_strategy_traceability(evidence: dict, source_payload_path: str) -> dict:
    evidence = dict(evidence)
    evidence["availability_profile"] = build_strategy_availability_profile(evidence)
    evidence["source_traceability"] = {
        "source_payload_path": source_payload_path,
        "source_tables": [
            "bank",
            "financial_summary",
            "climate_scenarios",
            "climate_risk_register",
            "value_chain_map",
            "climate_opportunities",
            "reporting_kpis",
            "business_model_impacts",
            "strategy_tradeoff_decisions",
        ],
        "top_risk_refs": [
            item.get("risk_id") or item.get("risk_name")
            for item in evidence.get("risk_summary", {}).get(
                "top_risks_by_financial_impact", []
            )
            if item.get("risk_id") or item.get("risk_name")
        ],
        "scenario_refs": [
            item.get("scenario_id")
            for item in [
                evidence.get("scenario_summary", {}).get("max_physical_risk_loss_pct_capital"),
                evidence.get("scenario_summary", {}).get("max_transition_risk_loss_pct_capital"),
                evidence.get("scenario_summary", {}).get("max_stranded_assets_estimate_meur"),
                evidence.get("scenario_summary", {}).get("max_revenue_at_risk_meur"),
            ]
            if isinstance(item, dict) and item.get("scenario_id")
        ],
        "value_chain_refs": [
            item.get("node_name")
            for item in evidence.get("value_chain_summary", {}).get(
                "largest_quantified_nodes", []
            )
            if item.get("node_name")
        ],
        "opportunity_refs": [
            item.get("opportunity_type") or item.get("description")
            for item in evidence.get("opportunity_summary", {}).get("examples", [])
            if item.get("opportunity_type") or item.get("description")
        ],
    }
    return evidence


strategy_source_path = (
    "strategy-specific payload"
    if strategy_payload is not payload
    else str(PAYLOAD_PATH)
)

strategy_evidence = add_strategy_traceability(
    strategy_evidence,
    strategy_source_path,
)

# Create a self-contained raw Strategy payload.
raw_strategy_payload = {
    key: strategy_payload.get(key)
    for key in [
        "metadata",
        "bank",
        "financial_summary",
        "climate_scenarios",
        "climate_risk_register",
        "value_chain_map",
        "climate_opportunities",
        "reporting_kpis",
        "business_model_impacts",
        "strategy_tradeoff_decisions",
        "value_chain_summary",
        "opportunity_summary",
        "portfolio_exposure_summary",
        "strategy_evidence_boundaries",
    ]
    if key in strategy_payload
}

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

RAW_STRATEGY_PATH = output_dir / "payload_BANK01_strategy_raw.json"
COMPACT_STRATEGY_PATH = output_dir / "compact_strategy_evidence_BANK01.json"

with open(RAW_STRATEGY_PATH, "w", encoding="utf-8") as f:
    json.dump(raw_strategy_payload, f, indent=2, ensure_ascii=False)

with open(COMPACT_STRATEGY_PATH, "w", encoding="utf-8") as f:
    json.dump(strategy_evidence, f, indent=2, ensure_ascii=False)

print("Strategy evidence prepared")
print(f"- Raw Strategy payload: {RAW_STRATEGY_PATH}")
print(f"- Compact Strategy evidence: {COMPACT_STRATEGY_PATH}")
print("- Availability profile:")
print(json.dumps(strategy_evidence["availability_profile"], indent=2, ensure_ascii=False))


In [ ]:

# ── STRATEGY STATE DEFINITION ───────────────────────────────
class StrategyState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict


In [ ]:

# ── STRATEGY REQUIREMENTS ───────────────────────────────────
# IFRS references are used internally only. Final text must NOT show IFRS paragraph references.

STRATEGY_REQUIREMENTS = """
STRICT STRATEGY DISCLOSURE REQUIREMENTS FOR THIS SECTION:

Final section title and headings must be exactly:
### Strategy
#### Climate-related risks and opportunities
#### Effects on business model and value chain
#### Effects on strategy and decision-making
#### Financial effects and resource allocation
#### Climate resilience and scenario analysis
#### Strategy limitations and evidence boundaries

Coverage requirements:
1. Identify climate-related risks and opportunities using evidence from the risk register, KPIs, targets and scenario analysis.
2. Distinguish physical risks from transition risks.
3. Cover short, medium and long-term horizons where scenario or risk-register evidence supports them.
4. Explain effects on business model and value chain using value_chain_map/value_chain_summary and business_model_impacts where available. If only partial financial quantification exists, state that boundary.
5. Explain effects on strategy and decision-making using evidence such as strategy_tradeoff_decisions, transition plan, net-zero target revision, climate scenario methodology, carbon credit budget, climate capex/opex, green loans and exposure metrics.
6. Include financial effects and resource allocation using numeric evidence: high-carbon exposure, fossil fuel exposure, climate capex, climate opex, revenue at risk, stranded assets, physical/transition loss percentages or financed emissions if relevant.
7. Include climate resilience and scenario analysis. Explain scenario families/types, frameworks, time horizons and key assumptions.
8. Do not claim the bank is resilient in general. Only describe resilience within the boundaries of the scenario evidence.
9. Do not claim Paris alignment unless directly supported by the evidence. If a scenario or target has a flag, explain it cautiously.
10. Do not overstate modelled estimates as actual losses.
11. Do not include visible IFRS paragraph references or bracketed IFRS tags in the final output.
12. If business model, value-chain, opportunity, trade-off, high-carbon exposure or fossil-fuel exposure evidence is present, use it. Do not state it is unavailable. If evidence is partial, disclose the boundary.
""".strip()


In [ ]:

# ── STRATEGY WRITER / JUDGE PROMPTS ─────────────────────────
STRATEGY_WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Strategy section of an IFRS S1/S2 aligned climate disclosure report for a commercial bank.

WRITING STANDARDS:
- Formal, third-person professional disclosure style.
- Evidence-based and audit-friendly.
- Do not use markdown tables.
- Do not include visible IFRS paragraph references, paragraph numbers, or bracketed IFRS tags.
- Do not invent strategy, value-chain, financial effect, or resilience claims.
- If evidence is missing, state the limitation in report-style language using "available evidence" or "source data". Avoid saying "payload" in final text.

STRICT RULES:
- Distinguish physical risk from transition risk.
- Distinguish modelled scenario outputs from actual financial impacts.
- Do not claim the bank is resilient overall. You may say the scenario evidence supports assessment within specified scenario boundaries.
- Do not claim Paris alignment unless directly supported by target or scenario flags, and even then use cautious wording.
- Avoid unsupported words: "ensures", "guarantees", "fully resilient", "compliant", "fully aligned", "proves".
- Keep board decision details concise if mentioned; Strategy is not Governance.
- Use value_chain_summary/value_chain_map when available; do not state value-chain evidence is unavailable when those fields are present.
- Use business_model_impacts when available to explain actual business model channels.
- Use climate_opportunities/opportunity_summary when available, but describe revenue impacts as estimates with confidence levels, not guaranteed revenue.
- Use high-carbon sector exposure and fossil-fuel exposure values when present; never say those data are unavailable if they exist in evidence.
- Use strategy_tradeoff_decisions if available, while noting whether quantified trade-off analysis is unavailable.
""".strip()


def build_strategy_writer_prompt(
    evidence: dict,
    judge_feedback: str | None = None,
) -> str:
    """
    Build an availability-aware Strategy prompt.

    The writer must use evidence that exists and state boundaries only where
    evidence is genuinely unavailable or partial.
    """
    profile = evidence.get("availability_profile", {})
    instructions = []

    if profile.get("physical_risk_evidence_available") and profile.get(
        "transition_risk_evidence_available"
    ):
        instructions.append(
            "- Distinguish and describe both physical and transition risks using the supplied examples."
        )
    else:
        instructions.append(
            "- Describe only the risk types evidenced; state the missing risk-type boundary without inventing risks."
        )

    if profile.get("time_horizons_available"):
        instructions.append(
            "- Use the available short-, medium-, and long-term horizons where supported."
        )

    if profile.get("business_model_detail_available"):
        instructions.append(
            "- Explain the supplied business-model impacts and transmission channels."
        )
    else:
        instructions.append(
            "- Business-model detail is incomplete. State the boundary once while still using available lending, operational, value-chain, and financial channels."
        )

    if profile.get("value_chain_detail_available"):
        instructions.append(
            "- Use the available value-chain mapping and material examples."
        )
        if profile.get("quantified_value_chain_exposure_available"):
            instructions.append(
                "- Include the available quantified value-chain exposures and identify qualitative-only nodes as a boundary."
            )
    else:
        instructions.append(
            "- Value-chain evidence is unavailable. Do not invent value-chain nodes or exposures."
        )

    if profile.get("strategy_tradeoff_evidence_available"):
        instructions.append(
            "- Describe only the documented Strategy trade-offs provided in the evidence."
        )
    else:
        instructions.append(
            "- No documented Strategy trade-off evidence is available. State the boundary once without inventing trade-offs."
        )

    if profile.get("resource_allocation_evidence_available"):
        instructions.append(
            "- Use climate capex and climate opex as resource-allocation evidence."
        )

    if profile.get("high_carbon_exposure_available"):
        instructions.append(
            "- Use the available high-carbon sector exposure metrics."
        )
    if profile.get("fossil_fuel_exposure_available"):
        instructions.append(
            "- Use the available fossil-fuel exposure metrics."
        )

    if profile.get("climate_opportunities_available"):
        instructions.append(
            "- Describe the concrete climate opportunities supplied in the evidence."
        )
        if profile.get("quantified_opportunity_estimates_available"):
            instructions.append(
                "- Use quantified opportunity revenue estimates cautiously and identify them as estimates, not guaranteed revenue."
            )
    else:
        instructions.append(
            "- No climate-opportunity evidence is available. Do not invent opportunities or revenue impacts."
        )

    if profile.get("scenario_analysis_available"):
        instructions.append(
            "- Explain the available scenario families, framework, horizons, assumptions, and key modelled outputs."
        )
    else:
        instructions.append(
            "- Scenario analysis is unavailable. Do not invent scenario results or resilience conclusions."
        )

    if profile.get("resilience_evidence_available"):
        instructions.append(
            "- Describe resilience only within the supplied scenario-specific assumptions and boundaries."
        )

    feedback_block = ""
    if judge_feedback:
        feedback_block = f"""
JUDGE FEEDBACK TO ADDRESS:
{judge_feedback}

REVISION POLICY:
- Fix omissions using available evidence.
- Preserve accurate evidence-boundary statements where evidence is unavailable.
- Never invent evidence to satisfy the judge.
""".strip()

    return f"""
{STRATEGY_REQUIREMENTS}

BANK:
{evidence['bank']['bank_name']} ({evidence['bank']['bank_id']})

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

EVIDENCE-AWARE INSTRUCTIONS:
{chr(10).join(instructions)}

COMPACT STRATEGY EVIDENCE — USE ONLY THIS DATA:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

GENERAL WRITING RULES:
- Write formal, publication-ready disclosure language.
- Use all material evidence that is available and relevant.
- When evidence is unavailable or partial, state the boundary once in the relevant subsection.
- Do not use the technical word "payload" in the final disclosure.
- Distinguish modelled scenario outputs from actual financial effects.
- Describe opportunity revenue impacts as estimates, not guaranteed revenue.
- Do not claim overall bank resilience or overall Paris alignment.
- Do not include visible IFRS paragraph references.
- Never invent risks, opportunities, trade-offs, value-chain nodes, assumptions, financial effects, or resilience conclusions.

{feedback_block}

Write the complete Strategy section only.
""".strip()


STRATEGY_JUDGE_SYSTEM = """
You are a strict IFRS S1/S2 Strategy disclosure reviewer and ESG audit specialist.
Your role is to identify genuine gaps, not to reward fluent writing.

Return ONLY valid JSON with this schema:
{
  "overall_score": 0-10,
  "evidence_support_score": 0-10,
  "ifrs_alignment_score": 0-10,
  "specificity_score": 0-10,
  "hallucination_risk": "low" | "medium" | "high",
  "approved": true | false,
  "checklist": {
    "all_six_subsections_present": true | false,
    "no_visible_ifrs_refs": true | false,
    "physical_transition_distinct": true | false,
    "time_horizons_covered": true | false,
    "business_model_effects_or_limitation": true | false,
    "value_chain_effects_or_limitation": true | false,
    "value_chain_data_used_if_available": true | false,
    "strategy_decision_making_covered": true | false,
    "financial_effects_covered": true | false,
    "high_carbon_fossil_exposure_used_if_available": true | false,
    "resource_allocation_covered": true | false,
    "opportunities_used_if_available": true | false,
    "tradeoffs_used_if_available": true | false,
    "scenario_analysis_covered": true | false,
    "scenario_assumptions_covered": true | false,
    "resilience_not_overclaimed": true | false,
    "paris_alignment_not_overclaimed": true | false,
    "modelled_estimates_not_overstated": true | false,
    "evidence_boundaries_present": true | false,
    "no_unsupported_strong_claims": true | false,
    "false_count": 0
  },
  "main_issues": [],
  "required_fixes": []
}

SCORING RULES:
- 9-10 only if strategy is directly evidenced and no material limitation is needed.
- 8 if strong but one material element is addressed only through limitation.
- 7 if two or more material elements are addressed through limitations, but no hallucination.
- 6 or lower if it overclaims resilience, Paris alignment, financial effects, or uses unsupported strong claims.
- A limitation statement reduces hallucination risk but does not count as full completeness.
- If evidence contains value_chain_summary/value_chain_map but the draft says value-chain evidence is unavailable, cap the score at 6.
- If evidence contains high-carbon and fossil-fuel exposure values but the draft says these data are unavailable, cap the score at 6.
- If evidence contains climate_opportunities but the draft does not mention concrete opportunities, cap the score at 7.
- If evidence contains strategy_tradeoff_decisions but the draft says no trade-off evidence is available, cap the score at 7.
""".strip()


def build_strategy_judge_prompt(draft: str, evidence: dict) -> str:
    """
    Build an availability-aware Strategy judge prompt.

    The judge must distinguish unavailable evidence from available evidence
    that the writer omitted, contradicted, or overstated.
    """
    profile = evidence.get("availability_profile", {})

    return f"""
Evaluate the Strategy draft against the compact evidence and its availability profile.

DRAFT:
{draft}

AVAILABILITY PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

COMPACT STRATEGY EVIDENCE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

EVALUATION PRINCIPLES:
1. Penalise claims that contradict evidence, invent unsupported information, or overstate estimates.
2. Penalise omission when material evidence is available but not used.
3. Do not demand evidence that is unavailable.
4. Correctly disclosed evidence boundaries reduce completeness but are not hallucinations.
5. Verify use of available physical and transition risks, time horizons, business-model channels, value-chain evidence, opportunities, exposure metrics, resource allocation, scenario assumptions, and scenario financial outputs.
6. Verify that opportunity revenue impacts are described as estimates.
7. Verify that scenario financial outputs are described as modelled estimates, not actual losses.
8. Verify that resilience claims remain within scenario-specific boundaries.
9. Verify that no overall Paris-alignment claim is made without direct evidence.
10. Do not reward fluent writing when available evidence is omitted or misrepresented.

SCORING:
- 9–10: materially complete, directly evidenced, and no material boundary is required.
- 8: strong with one material boundary or minor evidence-use issue.
- 7: usable with multiple correctly disclosed boundaries.
- 6: revision required because available evidence was omitted, contradicted, or overstated.
- 5 or below: major factual, evidence, or overclaiming failures.

Return valid JSON only. Keep all issue/fix strings concise (maximum 35 words each) and include no more than 6 items per array:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approval_status": "<approved|approved_with_limitations|revision_required|rejected>",
  "approved": <true if approved or approved_with_limitations, otherwise false>,
  "checklist": {{
    "required_structure_present": <true/false>,
    "physical_and_transition_risks_used_according_to_availability": <true/false>,
    "time_horizons_used_according_to_availability": <true/false>,
    "business_model_handled_according_to_availability": <true/false>,
    "value_chain_used_according_to_availability": <true/false>,
    "strategy_tradeoffs_handled_according_to_availability": <true/false>,
    "financial_effects_used_according_to_availability": <true/false>,
    "resource_allocation_used_according_to_availability": <true/false>,
    "high_carbon_and_fossil_exposure_used_if_available": <true/false>,
    "opportunities_used_according_to_availability": <true/false>,
    "scenario_analysis_used_according_to_availability": <true/false>,
    "scenario_assumptions_used_according_to_availability": <true/false>,
    "resilience_not_overclaimed": <true/false>,
    "paris_alignment_not_overclaimed": <true/false>,
    "modelled_estimates_not_overstated": <true/false>,
    "no_visible_ifrs_refs": <true/false>,
    "no_unsupported_strong_claims": <true/false>
  }},
  "available_evidence_omitted": [<specific available evidence omitted from the draft>],
  "unsupported_claims": [<specific unsupported claims>],
  "correctly_disclosed_evidence_boundaries": [<accurate boundary statements>],
  "main_issues": [<specific issues>],
  "required_fixes": [<actionable evidence-aware fixes>]
}}
""".strip()


print("Strategy prompts ready")


In [ ]:
# ── STRATEGY EVALUATION MODE ───────────────────────────────
# Deterministic rule checks and score caps have been removed.
# GPT-5.2 is solely responsible for evaluating, scoring, and approving
# the Strategy section against the compact evidence and judge prompt.

print("Strategy evaluation mode: GPT-5.2 judge only")


In [ ]:

# ── STRATEGY LANGGRAPH NODES ────────────────────────────────

def strategy_writer_node(state: StrategyState) -> StrategyState:
    is_revision = state["revision_count"] > 0
    feedback = None

    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n" +
            "\n".join(f"- {fix}" for fix in issues) +
            "\n\nFAILED CHECKLIST ITEMS:\n" +
            "\n".join(f"- {item}" for item in false_items)
        )

    prompt = build_strategy_writer_prompt(state["evidence"], judge_feedback=feedback)
    draft = call_writer_llm(
        system_prompt=STRATEGY_WRITER_SYSTEM,
        user_prompt=prompt,
    )

    print(f"\n{'='*50}")
    print(f"STRATEGY WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")

    return {**state, "draft": draft.strip(), "status": "judging"}


def strategy_judge_node(state: StrategyState) -> StrategyState:
    """
    Evaluate Strategy using only the GPT-5.2 judge.

    No deterministic heading checks, hard-fail overrides, score caps, or
    programmatic approval changes are applied.
    """
    draft = state["draft"]

    prompt = build_strategy_judge_prompt(draft, state["evidence"])
    judge_result = call_judge_llm_json(
        system_prompt=STRATEGY_JUDGE_SYSTEM,
        user_prompt=prompt,
    )

    # Preserve the judge's own decision. Add only safe defaults when omitted.
    judge_result.setdefault("approved", False)
    judge_result.setdefault(
        "approval_status",
        "approved" if judge_result.get("approved") else "revision_required",
    )
    judge_result.setdefault("main_issues", [])
    judge_result.setdefault("required_fixes", [])
    judge_result.setdefault("checklist", {})

    print("\nSTRATEGY JUDGE RESULT — GPT-5.2 ONLY")
    print(json.dumps(judge_result, indent=2, ensure_ascii=False))

    approved = bool(judge_result.get("approved"))
    status = "approved" if approved else "revising"

    return {
        **state,
        "judge_result": judge_result,
        "status": status,
    }


STRATEGY_REVISER_SYSTEM = """
You are a precise sustainability disclosure reviser.

Revise the existing Strategy section using only:
- the supplied compact evidence;
- the strict strategy requirements; and
- the judge's required fixes.

Rules:
- Fix every judge issue and failed checklist item.
- Preserve correct content that was not criticised.
- Never invent evidence.
- Clearly distinguish estimates, scenarios, and actual financial effects.
- Keep the exact six-subsection structure.
- Do not add visible IFRS paragraph references.
- Return only the complete revised Strategy section.
""".strip()


def strategy_reviser_node(state: StrategyState) -> StrategyState:
    if state["revision_count"] >= state["max_revisions"]:
        print("Max strategy revisions reached. Marking as failed.")
        return {**state, "status": "failed", "final_section": state["draft"]}

    judge = state.get("judge_result", {})
    issues = judge.get("required_fixes", [])
    checklist = judge.get("checklist", {})
    false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]

    revision_prompt = f"""
STRICT STRATEGY REQUIREMENTS:
{STRATEGY_REQUIREMENTS}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(state["evidence"].get("availability_profile", {}), indent=2, ensure_ascii=False)}

COMPACT STRATEGY EVIDENCE:
{json.dumps(state["evidence"], indent=2, ensure_ascii=False)}

REVISION BOUNDARY:
- Use available evidence when the judge identifies an omission.
- Preserve or improve accurate boundary statements where evidence is unavailable.
- Never invent missing opportunities, trade-offs, value-chain effects, scenario assumptions, financial impacts, or resilience conclusions.

CURRENT DRAFT:
{state["draft"]}

JUDGE REQUIRED FIXES:
{json.dumps(issues, indent=2, ensure_ascii=False)}

FAILED CHECKLIST ITEMS:
{json.dumps(false_items, indent=2, ensure_ascii=False)}

Revise the current draft and return only the complete revised Strategy section.
""".strip()

    revised_draft = call_reviser_llm(
        system_prompt=STRATEGY_REVISER_SYSTEM,
        user_prompt=revision_prompt,
    )

    new_revision_count = state["revision_count"] + 1
    print(f"\nStrategy revised with GPT-4.1 | revision {new_revision_count}")

    return {
        **state,
        "draft": revised_draft.strip(),
        "revision_count": new_revision_count,
        "status": "judging",
    }


def strategy_finalize_node(state: StrategyState) -> StrategyState:
    approved = bool(state.get("judge_result", {}).get("approved", False))
    return {
        **state,
        "final_section": state["draft"],
        "status": "approved" if approved else "failed",
    }


def strategy_should_continue(state: StrategyState) -> str:
    if state["status"] == "approved":
        return "finalize"
    if state.get("revision_count", 0) >= state.get("max_revisions", 1):
        return "finalize"
    return "reviser"

print("Strategy nodes ready")


In [ ]:

# ── BUILD AND COMPILE STRATEGY GRAPH ────────────────────────
strategy_builder = StateGraph(StrategyState)

strategy_builder.add_node("writer",   strategy_writer_node)
strategy_builder.add_node("judge",    strategy_judge_node)
strategy_builder.add_node("reviser",  strategy_reviser_node)
strategy_builder.add_node("finalize", strategy_finalize_node)

strategy_builder.add_edge(START, "writer")
strategy_builder.add_edge("writer", "judge")
strategy_builder.add_conditional_edges(
    "judge",
    strategy_should_continue,
    {
        "finalize": "finalize",
        "reviser": "reviser",
    }
)
strategy_builder.add_edge("reviser", "judge")
strategy_builder.add_edge("finalize", END)

strategy_graph = strategy_builder.compile()
print("Strategy graph compiled")


In [ ]:

# ── RUN STRATEGY SECTION ────────────────────────────────────
strategy_initial_state: StrategyState = {
    "bank_name":      bank_name,
    "evidence":       strategy_evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions":  1,  # one GPT-4.1 revision pass if the GPT-5.2 judge rejects the draft
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {},
}

print(f"Starting strategy generation for: {bank_name}\n")
strategy_result = strategy_graph.invoke(strategy_initial_state)


In [ ]:

# ── STRATEGY OUTPUT ─────────────────────────────────────────
print("\n" + "="*60)
print("FINAL STRATEGY JUDGE RESULT")
print("="*60)
print(json.dumps(strategy_result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("STRATEGY SECTION")
print("="*60)
print(strategy_result["final_section"])

# Save outputs
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "strategy_BANK01.md", "w", encoding="utf-8") as f:
    f.write(strategy_result["final_section"])

with open(output_dir / "strategy_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "strategy",
        "status":         strategy_result["status"],
        "approval_status": strategy_result["judge_result"].get("approval_status"),
        "raw_score":      strategy_result["judge_result"].get("raw_score_before_caps"),
        "final_score":    strategy_result["judge_result"].get("overall_score"),
        "score_cap_reason": strategy_result["judge_result"].get("score_cap_reason"),
        "revisions":      strategy_result["revision_count"],
        "approved":       strategy_result["judge_result"].get("approved"),
        "checklist":      strategy_result["judge_result"].get("checklist"),
        "issues":         strategy_result["judge_result"].get("main_issues"),
        "available_evidence_omitted": strategy_result["judge_result"].get("available_evidence_omitted"),
        "unsupported_claims": strategy_result["judge_result"].get("unsupported_claims"),
        "correctly_disclosed_evidence_boundaries": strategy_result["judge_result"].get("correctly_disclosed_evidence_boundaries"),
        "raw_strategy_payload_path": str(RAW_STRATEGY_PATH),
        "compact_strategy_evidence_path": str(COMPACT_STRATEGY_PATH),
    }, f, indent=2, ensure_ascii=False)

print("\nSaved to outputs/strategy_BANK01.md")

print(f"Raw Strategy payload saved to: {RAW_STRATEGY_PATH}")
print(f"Compact Strategy evidence saved to: {COMPACT_STRATEGY_PATH}")
